### Khởi tạo môi trường và nạp thư viện
Notebook thực hiện quy trình nạp, làm sạch bảng xếp hạng Spotify Weekly Charts, tổng hợp chỉ số hiệu suất bài hát và xây dựng biến mục tiêu phục vụ phân tích trực quan. Ô code này nạp các thư viện phân tích dữ liệu, bổ sung đường dẫn mã nguồn nội bộ và thiết lập hiển thị toàn bộ cột của bảng.


In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Bổ sung thư mục gốc vào đường dẫn để nạp module nội bộ
sys.path.append("..")

# Cấu hình hiển thị đầy đủ các cột của bảng
pd.set_option("display.max_columns", None)


### Nạp dữ liệu thô từ tập tin bảng xếp hạng
Dữ liệu thô từ file spotify-weekly-charts.csv được đọc theo từng chunk để tối ưu bộ nhớ và loại bỏ các dòng tiêu đề rác, chỉ giữ 29 cột cần thiết theo cấu hình loader. Ô code này nạp dữ liệu vào biến df_charts_raw và kiểm tra kích thước ban đầu.

In [2]:
from src.data_loader import load_charts_raw

# Nạp 29 cột được chọn lọc từ tập tin csv bảng xếp hạng thô
raw_charts_path = "../data/raw/spotify-weekly-charts.csv"
df_charts_raw = load_charts_raw(raw_charts_path, chunksize=500_000)

print(f"kích thước dữ liệu thô: {df_charts_raw.shape[0]:,} dòng, {df_charts_raw.shape[1]} cột")

[LOAD] Đang đọc charts từ ../data/raw/spotify-weekly-charts.csv (chunksize=500,000, 30 cột)...
  -> Chunk 1: 499,981 dòng (loại 19 header rác)
  -> Chunk 2: 499,981 dòng (loại 19 header rác)
  -> Chunk 3: 499,978 dòng (loại 22 header rác)
  -> Chunk 4: 287,986 dòng (loại 13 header rác)
[LOAD HOÀN THÀNH] Tổng thô: 1,787,999 | Loại rác: 73 | Còn lại: 1,787,926 dòng, 30 cột
kích thước dữ liệu thô: 1,787,926 dòng, 30 cột


### Quan sát các dòng đầu và cuối của dữ liệu thô
Bước kiểm tra cơ bản giúp xác nhận cấu trúc các trường thông tin và đảm bảo dữ liệu không bị cắt xén ở phần đầu hoặc phần cuối. Ô code này hiển thị 3 dòng đầu và 3 dòng cuối của bảng thô.


In [3]:
print("ba dòng đầu:")
display(df_charts_raw.head(3))

print("ba dòng cuối:")
display(df_charts_raw.tail(3))


ba dòng đầu:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
0,spotify:track:2gpQi3hbcUAcEG8m2dlgfB,1,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Plan A,2022-03-23,1,1,4,3003411,2022-04-14,0.583,0.834,0.0,1.0,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.934998,178203.0,Argentina,South America,Spanish
1,spotify:track:2x8oBuYaObjqHqgGuIUZ0b,2,WOS,1.0,WOS,spotify:artist:5YCc6xS5Gpj3EkaYGdjyNK,argentine indie,0,ARRANCARMELO,2022-04-06,2,129,2,2512175,2022-04-14,0.654,0.354,5.0,1.0,-7.358,0.0738,0.7240,0.0,0.1340,0.262,81.956001,183547.0,Argentina,South America,Spanish
2,spotify:track:2SJZdZ5DLtlRosJ2xHJJJa,3,Paulo Londra,1.0,Paulo Londra,spotify:artist:3vQ0GE3mI0dAaxIMYe5g7z,argentine hip hop,0,Chance,2022-04-06,3,59,2,2408983,2022-04-14,0.721,0.463,1.0,0.0,-9.483,0.0646,0.2410,0.0,0.0929,0.216,137.914993,204003.0,Argentina,South America,Spanish


ba dòng cuối:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
1787923,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,Elton John,spotify:artist:3PhoLpVuITZKcymswpck5b,mellow gold,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1.0,1.0,-6.32,0.0309,0.0354,0.000072,0.0915,0.934,116.031998,202735.0,Vietnam,Asia,Vietnamese
1787924,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,Dua Lipa,spotify:artist:6M2wZ9GZgrQXHCFfjv46we,pop,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1.0,1.0,-6.32,0.0309,0.0354,0.000072,0.0915,0.934,116.031998,202735.0,Vietnam,Asia,Vietnamese
1787925,spotify:track:6JIC3hbC28JZKZ8AlAqX8h,199,"Elton John, Dua Lipa, PNAU",3.0,PNAU,spotify:artist:6n28c9qs9hNGriNa72b26u,alternative dance,1,Cold Heart - PNAU Remix,2021-10-22,114,183,30,36817,2022-05-05,0.795,0.8,1.0,1.0,-6.32,0.0309,0.0354,0.000072,0.0915,0.934,116.031998,202735.0,Vietnam,Asia,Vietnamese


### Lấy mẫu ngẫu nhiên và kiểm tra kiểu dữ liệu
Việc lấy mẫu ngẫu nhiên hỗ trợ phát hiện các giá trị bất thường nằm rải rác trong tập dữ liệu lớn. Ô code này trích xuất 3 dòng ngẫu nhiên với hạt giống cố định và in thông tin tổng quát về kiểu dữ liệu của các cột.


In [4]:
print("ba dòng ngẫu nhiên:")
display(df_charts_raw.sample(3, random_state=42))

print("thông tin cấu trúc bảng:")
df_charts_raw.info()


ba dòng ngẫu nhiên:


,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,collab,track_name,release_date,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language
1701836,spotify:track:04gmklgYhNhZxRnAvSofoE,111,"pyrokinesis, MUKKA",2.0,MUKKA,spotify:artist:6a1CGYBFAUrrWZvD2iSsEi,russian alt pop,1,Днями-ночами,2020-03-19,111,133,33,33416,2021-11-11,0.564,0.878,9.0,1.0,-1.592,0.0556,0.179,0.000000,0.263,0.313,179.815002,197333.0,Ukraine,Europe,Ukrainian
1306232,spotify:track:2mM3gZ0BbPwPPMelbA8vgt,10,"Milly, Farruko, Nio Garcia, Jay Wheeler, Amenazzy",5.0,Milly,spotify:artist:7kXCcNRLEnblbTAppPraBp,reggaeton flow,1,No Te Enamores - Remix,2020-11-20,8,9,11,524204,2021-03-11,0.824,0.837,5.0,1.0,-4.603,0.1070,0.573,0.000017,0.271,0.580,100.021004,289933.0,Peru,South America,Spanish
450242,spotify:track:5wANPM4fQCJwkGd4rN57mH,78,Olivia Rodrigo,1.0,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,0,drivers license,2021-05-21,22,78,28,27820,2021-07-22,0.561,0.431,10.0,1.0,-8.810,0.0578,0.768,0.000014,0.106,0.137,143.875000,242013.0,El Salvador,Central America,Spanish


thông tin cấu trúc bảng:
<class 'pandas.DataFrame'>
RangeIndex: 1787926 entries, 0 to 1787925
Data columns (total 30 columns):
 #   Column             Dtype         
---  ------             -----         
 0   uri                str           
 1   rank               int32         
 2   artist_names       str           
 3   artists_num        str           
 4   artist_individual  str           
 5   artist_id          str           
 6   artist_genre       str           
 7   collab             str           
 8   track_name         str           
 9   release_date       str           
 10  peak_rank          int32         
 11  previous_rank      str           
 12  weeks_on_chart     int32         
 13  streams            int64         
 14  week               datetime64[us]
 15  danceability       float32       
 16  energy             float32       
 17  key                float32       
 18  mode               float32       
 19  loudness           float32       
 20  speechines

### Thống kê mô tả các biến định lượng
Bảng thống kê mô tả tóm tắt phân bố giá trị, độ phân tán và khoảng biến thiên của các biến số học. Ô code này tính toán thống kê mô tả và chuyển vị bảng để tiện theo dõi theo từng thuộc tính.


In [5]:
print("thống kê mô tả các biến số:")
display(df_charts_raw.describe().T)


thống kê mô tả các biến số:


,count,mean,min,25%,50%,75%,max,std
rank,1787926.0,99.068727,1.0,49.0,98.0,149.0,200.0,57.738891
peak_rank,1787926.0,40.074916,1.0,5.0,21.0,61.0,200.0,45.615156
weeks_on_chart,1787926.0,35.28783,1.0,7.0,20.0,46.0,290.0,42.665785
streams,1787926.0,697469.25577,2067.0,24859.0,64732.0,314269.0,84952932.0,2381626.600371
week,1787926,2021-09-14 12:10:38.313666,2016-12-29 00:00:00,2021-05-27 00:00:00,2021-10-14 00:00:00,2022-03-03 00:00:00,2022-07-14 00:00:00,NaN
danceability,1787531.0,0.698214,0.0859,0.613,0.721,0.797,0.986,0.131912
energy,1787531.0,0.656129,0.00316,0.557,0.673,0.775,1.0,0.156264
key,1787531.0,5.273468,0.0,2.0,5.0,8.0,11.0,3.616229
mode,1787531.0,0.551461,0.0,0.0,1.0,1.0,1.0,0.497345
loudness,1787531.0,-6.175125,-41.41,-7.508,-5.837,-4.516,1.906,2.386746


### Kiểm tra dòng trùng lặp và giá trị khuyết thiếu
Phát hiện các dòng lặp hoàn toàn và đếm số lượng giá trị thiếu trên từng thuộc tính để chuẩn bị cho các bước xử lý dữ liệu. Ô code này đếm dòng trùng lặp và liệt kê các cột có giá trị khuyết.


In [6]:
# Đếm số dòng trùng lặp hoàn toàn
dup_count = df_charts_raw.duplicated().sum()
print(f"số dòng trùng lặp hoàn toàn: {dup_count:,}")

# Thống kê các cột có giá trị khuyết thiếu
null_cols = df_charts_raw.isnull().sum()
null_cols = null_cols[null_cols > 0]
print("các cột có giá trị khuyết thiếu:")
display(null_cols)


số dòng trùng lặp hoàn toàn: 0
các cột có giá trị khuyết thiếu:


danceability        395
energy              395
key                 395
mode                395
loudness            395
speechiness         395
acousticness        395
instrumentalness    395
liveness            395
valence             395
tempo               395
duration            395
dtype: int64

### Xác định số lượng bài hát độc nhất theo mã URI
Cột uri là mã định danh gốc của Spotify dùng để phân biệt các bản thu âm trên bảng xếp hạng. Ô code này đếm số lượng giá trị duy nhất trong cột uri để nắm tổng số bài hát có mặt trong dữ liệu thô.


In [7]:
# Đếm số lượng uri duy nhất trong bảng thô
n_unique_uri = df_charts_raw["uri"].nunique()
print(f"số bài hát độc nhất theo uri: {n_unique_uri:,}")


số bài hát độc nhất theo uri: 39,202


### Tạo bản sao phục vụ làm sạch sâu
Bản sao độc lập giúp bảo toàn dữ liệu thô ban đầu để đối chiếu khi cần thiết trong suốt quá trình xử lý. Ô code này nhân bản df_charts_raw sang df_charts_clean và in số lượng dòng cột ban đầu.


In [8]:
# Tạo bản sao dữ liệu sạch
df_charts_clean = df_charts_raw.copy()
print(f"kích thước bảng làm sạch ban đầu: {df_charts_clean.shape[0]:,} dòng, {df_charts_clean.shape[1]} cột")


kích thước bảng làm sạch ban đầu: 1,787,926 dòng, 30 cột


### Chuẩn hóa mã định danh bài hát và nghệ sĩ
Tiền tố spotify:track: và spotify:artist: cần được loại bỏ để thu được mã ID thuần 22 ký tự, phục vụ liên kết với các bảng dữ liệu bổ trợ. Ô code này tạo cột track_id, xóa cột uri và ghi đè mã sạch vào artist_id.


In [9]:
# Tách lấy mã 22 ký tự cho bài hát và bỏ cột uri gốc
df_charts_clean["track_id"] = df_charts_clean["uri"].astype(str).str.split(":").str[-1]
df_charts_clean.drop(columns=["uri"], inplace=True)

# Chuẩn hóa mã nghệ sĩ ghi đè lên cột artist_id
df_charts_clean["artist_id"] = df_charts_clean["artist_id"].astype(str).str.split(":").str[-1]

print(f"độ dài track_id: {sorted(df_charts_clean['track_id'].str.len().unique().tolist())}")
print(f"độ dài artist_id: {sorted(df_charts_clean['artist_id'].str.len().unique().tolist())}")


độ dài track_id: [22]
độ dài artist_id: [22]


### Chuẩn hóa thời gian bảng xếp hạng và trích xuất năm tháng
Cột ngày xếp hạng tuần cần ở kiểu datetime để thực hiện các phép toán khoảng thời gian và phân tích chuỗi thời gian. Ô code này ép kiểu cột week sang datetime đồng thời trích xuất cột năm và tháng.


In [10]:
# Ép kiểu datetime cho cột tuần xếp hạng
df_charts_clean["week"] = pd.to_datetime(df_charts_clean["week"], errors="coerce")

# Trích xuất năm và tháng xếp hạng
df_charts_clean["year"] = df_charts_clean["week"].dt.year
df_charts_clean["month"] = df_charts_clean["week"].dt.month

print(f"khoảng thời gian week: từ {df_charts_clean['week'].min().date()} đến {df_charts_clean['week'].max().date()}")


khoảng thời gian week: từ 2016-12-29 đến 2022-07-14


### Chuẩn hóa ngày phát hành và tính độ trễ lọt bảng xếp hạng
Ngày phát hành gốc trên Spotify có nhiều định dạng khác nhau cần được xử lý an toàn để trích xuất năm phát hành và tính khoảng cách thời gian từ khi ra mắt đến tuần xếp hạng. Ô code này tạo cột release_date_clean, release_year và days_to_chart.


In [11]:
# Chuẩn hóa ngày phát hành sang kiểu datetime linh hoạt
df_charts_clean["release_date_clean"] = pd.to_datetime(
    df_charts_clean["release_date"], errors="coerce", format="mixed"
)

# Trích xuất 4 chữ số năm phát hành bằng biểu thức chính quy
df_charts_clean["release_year"] = pd.to_numeric(
    df_charts_clean["release_date"].astype(str).str.extract(r"^(\d{4})")[0],
    errors="coerce"
).astype("Int64")

# Tính số ngày từ lúc phát hành đến tuần xếp hạng
df_charts_clean["days_to_chart"] = (
    df_charts_clean["week"] - df_charts_clean["release_date_clean"]
).dt.days

print(f"số giá trị thiếu ở release_date_clean: {df_charts_clean['release_date_clean'].isna().sum():,}")
print(f"số giá trị thiếu ở release_year: {df_charts_clean['release_year'].isna().sum():,}")


số giá trị thiếu ở release_date_clean: 0
số giá trị thiếu ở release_year: 0


### Lọc bỏ các dòng thiếu đặc trưng âm học
Bộ 12 chỉ số âm thanh là các biến dự báo cốt lõi của đề tài nên các dòng thiếu bất kỳ chỉ số nào trong nhóm này sẽ bị loại bỏ. Ô code này xác định các dòng khuyết audio features và lọc bỏ khỏi bảng làm sạch.


In [12]:
# Danh sách 12 đặc trưng âm học chuẩn
audio_cols = [
    "danceability", "energy", "key", "mode", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration"
]

# Lọc bỏ các dòng khuyết thiếu đặc trưng âm học
missing_audio = df_charts_clean[audio_cols].isna().any(axis=1)
n_before = len(df_charts_clean)
df_charts_clean = df_charts_clean[~missing_audio].copy()

print(f"số dòng bị loại do thiếu audio feature: {missing_audio.sum():,}")
print(f"số dòng còn lại: {len(df_charts_clean):,} (từ {n_before:,})")


số dòng bị loại do thiếu audio feature: 395
số dòng còn lại: 1,787,531 (từ 1,787,926)


### Chuẩn hóa thể loại âm nhạc của nghệ sĩ
Cột thể loại trong dữ liệu bảng xếp hạng chứa chuỗi thể loại đơn lẻ hoặc ký tự 0 đại diện cho trường hợp khuyết thiếu. Ô code này quy đổi các giá trị không hợp lệ về unknown và đưa các thể loại còn lại về chữ thường.


In [13]:
# Hàm chuyển đổi thể loại về dạng chuẩn
def parse_genre(g_val):
    if pd.isna(g_val):
        return "unknown"
    s = str(g_val).strip()
    if s in ("0", "0.0", "", "none", "nan", "null", "unknown"):
        return "unknown"
    return s.lower()

# Ánh xạ qua tập giá trị duy nhất để tăng tốc độ xử lý
unique_genres = df_charts_clean["artist_genre"].unique()
genre_map = {g: parse_genre(g) for g in unique_genres}
df_charts_clean["artist_genre"] = df_charts_clean["artist_genre"].map(genre_map).fillna("unknown")

n_unknown = (df_charts_clean["artist_genre"] == "unknown").sum()
print(f"số dòng unknown: {n_unknown:,} dòng ({n_unknown / len(df_charts_clean) * 100:.2f}%)")
print(f"số thể loại độc nhất: {df_charts_clean['artist_genre'].nunique():,}")
print("top 10 thể loại phổ biến:")
display(df_charts_clean["artist_genre"].value_counts().head(10))


số dòng unknown: 114,228 dòng (6.39%)
số thể loại độc nhất: 1,874
top 10 thể loại phổ biến:


artist_genre
pop             156902
unknown         114228
trap latino      86148
reggaeton        73510
latin            73194
dance pop        49378
rap              31601
uk pop           27131
canadian pop     20416
k-pop            19415
Name: count, dtype: int64

### Đổi tên các trường địa lý và ngôn ngữ bảng xếp hạng
Tên các cột country, region và language được gắn thêm tiền tố chart để phân biệt rõ ràng với thông tin địa lý và phân vùng kinh tế từ World Bank khi thực hiện ghép nối sau này. Ô code này đổi tên ba cột trên.

In [14]:
# Đổi tên cột địa lý của bảng xếp hạng
df_charts_clean.rename(
    columns={
        "country": "chart_country",
        "region": "chart_region",
        "language": "chart_language"
    },
    inplace=True
)

print("các cột địa lý sau khi đổi tên:")
display(df_charts_clean[["chart_country", "chart_region"]].head(3))


các cột địa lý sau khi đổi tên:


,chart_country,chart_region
0,Argentina,South America
1,Argentina,South America
2,Argentina,South America


### Tối ưu kiểu dữ liệu và tách cờ bài hát mới
Cột previous_rank chứa giá trị chuỗi với -1 đại diện cho bài hát mới lọt bảng xếp hạng. Ô code này tạo cờ nhị phân is_new_entry, ép kiểu số nguyên có thể khuyết cho previous_rank, chuyển key và mode sang kiểu số nguyên nhỏ, và chuyển các cột phân loại sang category để giảm bộ nhớ.

In [15]:
# Tạo cờ bài hát mới lọt bảng xếp hạng và chuẩn hóa previous_rank
df_charts_clean["is_new_entry"] = (df_charts_clean["previous_rank"].astype(str) == "-1") | df_charts_clean["previous_rank"].isnull()
df_charts_clean["previous_rank"] = pd.to_numeric(df_charts_clean["previous_rank"].replace("-1", np.nan), errors="coerce").astype("Int16")

# Ép kiểu số nguyên an toàn cho key, mode và artists_num
df_charts_clean["key"] = pd.to_numeric(df_charts_clean["key"], errors="coerce").astype("Int8")
df_charts_clean["mode"] = pd.to_numeric(df_charts_clean["mode"], errors="coerce").astype("Int8")
df_charts_clean["artists_num"] = pd.to_numeric(df_charts_clean["artists_num"], errors="coerce").fillna(1).astype("int16")

# Chuyển đổi các cột chuỗi lặp lại sang kiểu category để tiết kiệm bộ nhớ
cat_cols = ["chart_country", "chart_region", "chart_language", "artist_genre"]
for c in cat_cols:
    df_charts_clean[c] = df_charts_clean[c].astype("category")

# Giải phóng bộ nhớ của bảng thô
del df_charts_raw
print("đã tối ưu kiểu dữ liệu cho previous_rank, key, mode và chuyển 4 cột sang category")

đã tối ưu kiểu dữ liệu cho previous_rank, key, mode và chuyển 4 cột sang category


### Lựa chọn và sắp xếp các cột theo thứ tự chuẩn
Bảng dữ liệu làm sạch được tinh gọn về đúng 34 cột chuẩn hóa và sắp xếp theo trình tự mạch lạc giữa các nhóm thông tin. Ô code này kiểm tra tính đầy đủ của các trường và định vị lại cấu trúc cột.

In [16]:
# Danh sách 34 cột chuẩn hóa theo thứ tự logic
cols_to_keep = [
    # Nhóm định danh bài hát và nghệ sĩ
    "track_id", "track_name", "artist_id", "artist_names", "artist_individual", "artists_num", "artist_genre",
    # Nhóm thời gian và địa lý bảng xếp hạng
    "week", "year", "month", "chart_country", "chart_region", "chart_language",
    # Nhóm thành tích và lượt nghe
    "rank", "previous_rank", "is_new_entry", "peak_rank", "weeks_on_chart", "streams",
    # Nhóm phát hành và độ tuổi ca khúc
    "release_date", "release_date_clean", "release_year", "days_to_chart",
    # Nhóm 12 đặc trưng âm học
    "danceability", "energy", "key", "mode", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration"
]

missing_cols = [c for c in cols_to_keep if c not in df_charts_clean.columns]
assert len(missing_cols) == 0, f"thiếu các cột: {missing_cols}"

df_charts_clean = df_charts_clean[cols_to_keep].copy()
print(f"số cột được giữ lại: {len(df_charts_clean.columns)}")

số cột được giữ lại: 35


### Tổng kiểm tra bảng làm sạch sau phần hai
Kiểm tra toàn diện nhằm xác minh tính hợp lệ của độ dài mã định danh, tình trạng khuyết thiếu và số lượng bài hát độc nhất trước khi chuyển sang phân tách dữ liệu. Ô code này in thông số tổng quan và 3 dòng đầu.


In [17]:
print(f"kích thước bảng sau làm sạch: {df_charts_clean.shape[0]:,} dòng, {df_charts_clean.shape[1]} cột")
print(f"track_id đều đủ 22 ký tự: {(df_charts_clean['track_id'].str.len() == 22).all()}")
print(f"artist_id đều đủ 22 ký tự: {(df_charts_clean['artist_id'].str.len() == 22).all()}")
print(f"số track_id độc nhất: {df_charts_clean['track_id'].nunique():,}")

print("các cột còn giá trị khuyết thiếu:")
display(df_charts_clean.isnull().sum()[df_charts_clean.isnull().sum() > 0])

print("ba dòng đầu của bảng làm sạch:")
display(df_charts_clean.head(3))


kích thước bảng sau làm sạch: 1,787,531 dòng, 35 cột
track_id đều đủ 22 ký tự: True
artist_id đều đủ 22 ký tự: True
số track_id độc nhất: 39,190
các cột còn giá trị khuyết thiếu:


previous_rank    167519
dtype: int64

ba dòng đầu của bảng làm sạch:


,track_id,track_name,artist_id,artist_names,artist_individual,artists_num,artist_genre,week,year,month,chart_country,chart_region,chart_language,rank,previous_rank,is_new_entry,peak_rank,weeks_on_chart,streams,release_date,release_date_clean,release_year,days_to_chart,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration
0,2gpQi3hbcUAcEG8m2dlgfB,Plan A,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,Paulo Londra,1,argentine hip hop,2022-04-14,2022,4,Argentina,South America,Spanish,1,1,False,1,4,3003411,2022-03-23,2022-03-23,2022,22,0.583,0.834,0,1,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.934998,178203.0
1,2x8oBuYaObjqHqgGuIUZ0b,ARRANCARMELO,5YCc6xS5Gpj3EkaYGdjyNK,WOS,WOS,1,argentine indie,2022-04-14,2022,4,Argentina,South America,Spanish,2,129,False,2,2,2512175,2022-04-06,2022-04-06,2022,8,0.654,0.354,5,1,-7.358,0.0738,0.7240,0.0,0.1340,0.262,81.956001,183547.0
2,2SJZdZ5DLtlRosJ2xHJJJa,Chance,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,Paulo Londra,1,argentine hip hop,2022-04-14,2022,4,Argentina,South America,Spanish,3,59,False,3,2,2408983,2022-04-06,2022-04-06,2022,8,0.721,0.463,1,0,-9.483,0.0646,0.2410,0.0,0.0929,0.216,137.914993,204003.0


### Tách bảng xếp hạng toàn cầu và bảng quốc gia
Bảng xếp hạng Global phản ánh thị trường toàn thế giới với quy mô stream vượt trội và thời gian theo dõi dài hơn, trong khi bảng quốc gia thể hiện độ phủ thị trường địa phương. Ô code này tách hai tập dữ liệu độc lập và kiểm tra tổng số dòng.


In [18]:
# Tách thành hai bảng độc lập theo thị trường
is_global = df_charts_clean["chart_country"] == "Global"
df_charts_global = df_charts_clean[is_global].copy()
df_charts_country = df_charts_clean[~is_global].copy()

# Loại bỏ các category không còn xuất hiện sau khi phân tách
cat_cols = ["chart_country", "chart_region", "chart_language", "artist_genre"]
for c in cat_cols:
    if c in df_charts_country.columns and hasattr(df_charts_country[c], "cat"):
        df_charts_country[c] = df_charts_country[c].cat.remove_unused_categories()
    if c in df_charts_global.columns and hasattr(df_charts_global[c], "cat"):
        df_charts_global[c] = df_charts_global[c].cat.remove_unused_categories()

print(f"tổng: {len(df_charts_clean):,} dòng")
print(f"global: {len(df_charts_global):,} dòng, {df_charts_global['track_id'].nunique():,} bài, {df_charts_global['week'].nunique():,} tuần")
print(f"quốc gia: {len(df_charts_country):,} dòng, {df_charts_country['track_id'].nunique():,} bài, {df_charts_country['chart_country'].nunique():,} quốc gia")
assert len(df_charts_global) + len(df_charts_country) == len(df_charts_clean)

tổng: 1,787,531 dòng
global: 95,932 dòng, 5,347 bài, 290 tuần
quốc gia: 1,691,599 dòng, 35,950 bài, 73 quốc gia


### Khảo sát phạm vi thời gian và mức độ giao thoa bài hát
So sánh khoảng thời gian bắt đầu và kết thúc giữa hai tập dữ liệu, đồng thời thống kê số bài hát chỉ xuất hiện trên bảng toàn cầu, chỉ có ở quốc gia hoặc xuất hiện ở cả hai. Ô code này tính toán tập hợp bài hát giữa hai thị trường.


In [19]:
print(f"chart_region của global: {df_charts_global['chart_region'].unique().tolist()}")
print(f"thời gian global: từ {df_charts_global['week'].min().date()} đến {df_charts_global['week'].max().date()}")
print(f"thời gian quốc gia: từ {df_charts_country['week'].min().date()} đến {df_charts_country['week'].max().date()}")

tracks_global = set(df_charts_global["track_id"])
tracks_country = set(df_charts_country["track_id"])

both = tracks_global & tracks_country
only_global = tracks_global - tracks_country
only_country = tracks_country - tracks_global

print(f"có ở cả hai: {len(both):,} bài")
print(f"chỉ có ở global: {len(only_global):,} bài")
print(f"chỉ có ở quốc gia: {len(only_country):,} bài")


chart_region của global: ['Global']
thời gian global: từ 2016-12-29 đến 2022-07-14
thời gian quốc gia: từ 2021-02-04 đến 2022-07-14
có ở cả hai: 2,107 bài
chỉ có ở global: 3,240 bài
chỉ có ở quốc gia: 33,843 bài


### Đánh giá hệ số nhân dòng theo nghệ sĩ
Do cấu trúc dữ liệu ghi nhận mỗi nghệ sĩ trong bài hát trên một dòng riêng, số dòng thực tế bị nhân lên so với số vị trí xếp hạng tuần. Ô code này tính toán tỷ lệ dòng trên số vị trí duy nhất trong bảng quốc gia.


In [20]:
# Tính số vị trí xếp hạng duy nhất theo quốc gia, tuần và thứ hạng
n_rows_country = len(df_charts_country)
n_unique_slots = df_charts_country.drop_duplicates(
    subset=["chart_country", "week", "rank"]
).shape[0]
multiplier = n_rows_country / n_unique_slots

print(f"số dòng bảng quốc gia: {n_rows_country:,}")
print(f"số vị trí xếp hạng duy nhất: {n_unique_slots:,}")
print(f"hệ số nhân theo nghệ sĩ: {multiplier:.2f}")


số dòng bảng quốc gia: 1,691,599
số vị trí xếp hạng duy nhất: 1,051,382
hệ số nhân theo nghệ sĩ: 1.61


### Kiểm tra tính duy nhất của khóa hạt bảng xếp hạng
Khóa hạt xác định một bản ghi duy nhất trong bảng xếp hạng là bộ bốn trường gồm mã bài hát, mã nghệ sĩ, quốc gia và ngày chốt tuần. Ô code này kiểm tra số lượng dòng trùng lặp theo khóa hạt trên toàn bộ bảng dữ liệu làm sạch.

In [21]:
# Kiểm tra tính duy nhất của bộ khóa hạt trên toàn bộ bảng xếp hạng
chart_grain_cols = ["track_id", "artist_id", "chart_country", "week"]
dup_grain_count = df_charts_clean.duplicated(subset=chart_grain_cols).sum()

print(f"số lượng dòng trùng lặp theo khóa hạt: {dup_grain_count:,}")
assert dup_grain_count == 0, "cảnh báo: có trùng lặp theo khóa hạt bảng xếp hạng"

số lượng dòng trùng lặp theo khóa hạt: 0


### Kiểm chứng số dòng nhân bản theo số lượng nghệ sĩ hợp tác
Mỗi vị trí xếp hạng của bài hát được nhân bản số dòng bằng đúng số nghệ sĩ tham gia biểu diễn. Ô code này so sánh số dòng thực tế của từng bài hát trong tuần với biến artists_num để chứng minh bản chất của hệ số nhân 1.61.

In [22]:
# Đếm số dòng thực tế của từng bài hát theo quốc gia và tuần
df_group_sizes = df_charts_clean.groupby(["track_id", "chart_country", "week"]).size().reset_index(name="group_size")
df_first_slots = df_charts_clean.drop_duplicates(subset=["track_id", "chart_country", "week"])
df_slots_check = df_first_slots.merge(df_group_sizes, on=["track_id", "chart_country", "week"])

# Xác định các nhóm có số dòng thực tế khác biệt với artists_num
mismatches = df_slots_check[df_slots_check["group_size"] != df_slots_check["artists_num"]]
print(f"số nhóm có số dòng khác artists_num: {len(mismatches)} (do nghệ sĩ n/a)")
display(mismatches[["track_id", "track_name", "artist_names", "chart_country", "week", "rank", "artists_num", "group_size"]])

assert len(mismatches) == 2, "cảnh báo: số lượng mismatch khác 2 nhóm Bodø Domkor đã biết"
assert (mismatches["track_id"] == "0GEeVI8Wa1KXGTStgBUerG").all(), "cảnh báo: xuất hiện bài hát khác ngoài Bodø Domkor"
print("xác nhận 100% các bài hát khác đều có số dòng trùng khớp tuyệt đối với artists_num")

số nhóm có số dòng khác artists_num: 2 (do nghệ sĩ n/a)


,track_id,track_name,artist_names,chart_country,week,rank,artists_num,group_size
751634,0GEeVI8Wa1KXGTStgBUerG,"Ja, Vi Elsker Dette Landet","N/A, Bodø Domkor",Norway,2022-05-19,166,2,1
752282,0GEeVI8Wa1KXGTStgBUerG,"Ja, Vi Elsker Dette Landet","N/A, Bodø Domkor",Norway,2021-05-20,184,2,1


xác nhận 100% các bài hát khác đều có số dòng trùng khớp tuyệt đối với artists_num


### Kiểm tra ngày chốt bảng xếp hạng cố định vào thứ Năm
Toàn bộ các chu kỳ xếp hạng của Spotify được chốt vào thứ Năm hàng tuần làm căn cứ đồng bộ với dữ liệu COVID-19. Ô code này kiểm tra ngày trong tuần của toàn bộ các dòng dữ liệu để đảm bảo tính đồng nhất thời gian.

In [23]:
# Kiểm tra tất cả các giá trị thời gian week có phải là thứ Năm không
non_thursday_count = (df_charts_clean["week"].dt.day_name() != "Thursday").sum()

print(f"số dòng có ngày chốt bảng xếp hạng khác thứ Năm: {non_thursday_count:,}")
assert non_thursday_count == 0, "cảnh báo: tồn tại tuần không chốt vào thứ Năm"

số dòng có ngày chốt bảng xếp hạng khác thứ Năm: 0


### Khảo sát số tuần xếp hạng và các vị trí khuyết ở bảng quốc gia
Số vị trí thực tế của bảng quốc gia chênh lệch so với tích số lý thuyết do thời điểm ra mắt bảng xếp hạng ở các nước khác nhau và một số tuần khuyết vị trí. Ô code này thống kê các quốc gia có ít hơn 76 tuần và các tuần chỉ có 199 thứ hạng.

In [24]:
# Thống kê các quốc gia ra mắt muộn có ít hơn 76 tuần trên bảng xếp hạng
weeks_per_country = df_charts_country.groupby("chart_country")["week"].nunique()
incomplete_countries = weeks_per_country[weeks_per_country < 76]
print(f"số quốc gia có ít hơn 76 tuần ({len(incomplete_countries)} quốc gia):")
display(incomplete_countries.to_frame(name="so_tuan_xep_hang"))

# Thống kê số lượng cặp quốc gia và tuần có ít hơn 200 vị trí thứ hạng
slots_per_cw = df_charts_country.drop_duplicates(
    subset=["chart_country", "week", "rank"]
).groupby(["chart_country", "week"])["rank"].count()
missing_slots_cw = slots_per_cw[slots_per_cw < 200]
print(f"số cặp quốc gia và tuần có ít hơn 200 thứ hạng: {len(missing_slots_cw):,}")

số quốc gia có ít hơn 76 tuần (8 quốc gia):


,so_tuan_xep_hang
chart_country,
Belarus,22
Kazakhstan,22
Nigeria,22
Norway,75
Pakistan,10
Philippines,75
United Arab Emirates,75
Venezuela,17


số cặp quốc gia và tuần có ít hơn 200 thứ hạng: 217


### Kiểm tra tính bất biến của đặc trưng âm học theo bài hát
Các đặc trưng âm học và ngày phát hành là thuộc tính cố định của ca khúc và không được thay đổi giữa các tuần hoặc các quốc gia. Ô code này kiểm tra số lượng giá trị phân biệt của từng đặc trưng trên từng bài hát.

In [25]:
# Kiểm tra số lượng giá trị duy nhất của từng đặc trưng theo track_id
audio_check_cols = [
    "danceability", "energy", "key", "mode", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration",
    "release_date"
]
nunique_audio = df_charts_clean.groupby("track_id")[audio_check_cols].nunique()
varying_audio_count = (nunique_audio.max() > 1).sum()

print(f"số đặc trưng âm học có biến động theo dòng cùng bài hát: {varying_audio_count}")
assert varying_audio_count == 0, "cảnh báo: đặc trưng âm học thay đổi giữa các dòng của cùng bài hát"

số đặc trưng âm học có biến động theo dòng cùng bài hát: 0


### Xây dựng bảng thuộc tính bài hát duy nhất
Bảng bài hát lưu trữ thông tin nhận dạng và 12 đặc trưng âm học không đổi của từng bài, loại bỏ nhân bản bằng cách giữ dòng đầu tiên. Thể loại artist_genre trong bảng này là thể loại của nghệ sĩ đầu tiên chứ không phải thể loại độc lập của bài hát. Ô code này trích xuất df_tracks từ df_charts_clean và kiểm tra tính duy nhất.

In [26]:
# Trích xuất thông tin độc nhất của từng bài hát
track_dim_cols = [
    "track_id", "track_name", "artist_id", "artist_names", "artists_num", "artist_genre",
    "release_date", "release_date_clean", "release_year",
    "danceability", "energy", "key", "mode", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration"
]

df_tracks = df_charts_clean[track_dim_cols].drop_duplicates(
    subset=["track_id"], keep="first"
)
print(f"số bài hát trong df_tracks: {len(df_tracks):,}")
assert df_tracks["track_id"].is_unique, "cảnh báo: track_id trong df_tracks không duy nhất"
display(df_tracks.head(3))


số bài hát trong df_tracks: 39,190


,track_id,track_name,artist_id,artist_names,artists_num,artist_genre,release_date,release_date_clean,release_year,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration
0,2gpQi3hbcUAcEG8m2dlgfB,Plan A,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,1,argentine hip hop,2022-03-23,2022-03-23,2022,0.583,0.834,0,1,-4.875,0.0444,0.0495,0.0,0.0658,0.557,173.934998,178203.0
1,2x8oBuYaObjqHqgGuIUZ0b,ARRANCARMELO,5YCc6xS5Gpj3EkaYGdjyNK,WOS,1,argentine indie,2022-04-06,2022-04-06,2022,0.654,0.354,5,1,-7.358,0.0738,0.7240,0.0,0.1340,0.262,81.956001,183547.0
2,2SJZdZ5DLtlRosJ2xHJJJa,Chance,3vQ0GE3mI0dAaxIMYe5g7z,Paulo Londra,1,argentine hip hop,2022-04-06,2022-04-06,2022,0.721,0.463,1,0,-9.483,0.0646,0.2410,0.0,0.0929,0.216,137.914993,204003.0


### Xuất bảng thuộc tính bài hát duy nhất ra tập tin csv
Bảng df_tracks lưu trữ thông tin nhận dạng và 12 đặc trưng âm học không đổi của từng ca khúc. Ô code này tạo thư mục lưu trữ nếu chưa có và xuất dữ liệu ra tập tin tracks_dimension.csv.

In [27]:
import os

# Tạo thư mục lưu trữ dữ liệu đã xử lý nếu chưa tồn tại
os.makedirs("../data/processed", exist_ok=True)

# Xuất bảng bài hát duy nhất phục vụ làm giàu dữ liệu và mô hình hóa
tracks_out_path = "../data/processed/tracks_dimension.csv"
df_tracks.to_csv(tracks_out_path, index=False)

print(f"đã xuất dữ liệu bài hát: {len(df_tracks):,} dòng ra {tracks_out_path}")

đã xuất dữ liệu bài hát: 39,190 dòng ra ../data/processed/tracks_dimension.csv


### Khảo sát cấu trúc bảng xếp hạng toàn cầu
Bảng xếp hạng Global phản ánh thị trường toàn cầu với 290 tuần liên tục từ năm 2016 đến năm 2022. Dữ liệu này được duy trì trong bộ nhớ phục vụ phân tích vĩ mô mà không cần xuất riêng tập tin đĩa.

In [28]:
# Kiểm tra tổng thể bảng xếp hạng toàn cầu được lưu trong bộ nhớ
print(f"kích thước bảng xếp hạng toàn cầu: {len(df_charts_global):,} dòng, {df_charts_global['track_id'].nunique():,} bài hát")
display(df_charts_global[["track_name", "artist_names", "week", "rank", "streams"]].head(3))

kích thước bảng xếp hạng toàn cầu: 95,932 dòng, 5,347 bài hát


,track_name,artist_names,week,rank,streams
573695,Wow.,Post Malone,2019-01-17,2,29944242
573696,"thank u, next",Ariana Grande,2019-01-17,3,27807232
573697,Without Me,Halsey,2019-01-17,4,27138512


### Xuất bảng xếp hạng làm sạch tổng thể ra định dạng parquet
Bảng df_charts_clean chứa toàn bộ dữ liệu bảng xếp hạng chuẩn hóa sau khi đã loại các dòng thiếu đặc trưng âm học. Định dạng Parquet tối ưu dung lượng (khoảng 65MB thay vì 470MB của CSV) và bảo toàn chuẩn xác kiểu dữ liệu.

In [29]:
# Xuất toàn bộ bảng xếp hạng sạch ra định dạng parquet tối ưu dung lượng
charts_clean_parquet_path = "../data/processed/charts_clean.parquet"
df_charts_clean.to_parquet(charts_clean_parquet_path, index=False)

print(f"đã xuất bảng xếp hạng sạch: {len(df_charts_clean):,} dòng ra {charts_clean_parquet_path}")

đã xuất bảng xếp hạng sạch: 1,787,531 dòng ra ../data/processed/charts_clean.parquet


### Nạp dữ liệu thô từ tập tin phân loại mã quốc gia
Tập tin country_codes.csv chứa bảng phân loại khu vực địa lý và nhóm thu nhập kinh tế của World Bank theo từng mã quốc gia. Ô code này nạp dữ liệu thô vào biến df_country_codes_raw.


In [30]:
from src.data_loader import load_country_codes_raw

# Nạp dữ liệu mã quốc gia và phân loại thu nhập World Bank
raw_country_codes_path = "../data/raw/country_codes.csv"
df_country_codes_raw = load_country_codes_raw(raw_country_codes_path)

print(f"kích thước dữ liệu mã quốc gia thô: {df_country_codes_raw.shape[0]:,} dòng, {df_country_codes_raw.shape[1]} cột")

kích thước dữ liệu mã quốc gia thô: 217 dòng, 3 cột


### Quan sát các dòng đầu và cuối của bảng mã quốc gia
Kiểm tra trực quan giúp xác minh cấu trúc 3 cột mã định danh, vùng lãnh thổ và nhóm thu nhập kinh tế. Ô code này hiển thị 3 dòng đầu và 3 dòng cuối của bảng.


In [31]:
print("ba dòng đầu:")
display(df_country_codes_raw.head(3))

print("ba dòng cuối:")
display(df_country_codes_raw.tail(3))

ba dòng đầu:


,country_code,region,income_group
0,ABW,Latin America & Caribbean,High income
1,AFG,South Asia,Low income
2,AGO,Sub-Saharan Africa,Lower middle income


ba dòng cuối:


,country_code,region,income_group
214,ZAF,Sub-Saharan Africa,Upper middle income
215,ZMB,Sub-Saharan Africa,Lower middle income
216,ZWE,Sub-Saharan Africa,Lower middle income


### Kiểm tra cấu trúc dữ liệu và phân bố danh mục phân loại
Xem xét phân bố các quốc gia theo từng khu vực địa lý và mức thu nhập của World Bank. Ô code này in cấu trúc bảng và bảng đếm tần suất cho hai biến phân loại.


In [32]:
print("thông tin cấu trúc bảng mã quốc gia:")
df_country_codes_raw.info()

print("phân bố khu vực địa lý:")
display(df_country_codes_raw["region"].value_counts())

print("phân bố nhóm thu nhập:")
display(df_country_codes_raw["income_group"].value_counts())

thông tin cấu trúc bảng mã quốc gia:
<class 'pandas.DataFrame'>
RangeIndex: 217 entries, 0 to 216
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   country_code  217 non-null    str  
 1   region        217 non-null    str  
 2   income_group  217 non-null    str  
dtypes: str(3)
memory usage: 13.4 KB
phân bố khu vực địa lý:


region
Europe & Central Asia         58
Sub-Saharan Africa            48
Latin America & Caribbean     42
East Asia & Pacific           37
Middle East & North Africa    21
South Asia                     8
North America                  3
Name: count, dtype: int64

phân bố nhóm thu nhập:


income_group
High income            82
Upper middle income    55
Lower middle income    54
Low income             26
Name: count, dtype: int64

### Kiểm tra trùng lặp và tính toàn vẹn của mã quốc gia
Mã quốc gia ISO-3 là khóa liên kết quan trọng, cần đảm bảo không có dòng trùng lặp và không có giá trị khuyết thiếu. Ô code này kiểm tra trùng lặp và xác minh độ dài mã 3 ký tự.


In [33]:
# Kiểm tra dòng trùng lặp và giá trị khuyết thiếu
dup_codes = df_country_codes_raw.duplicated(subset=["country_code"]).sum()
null_count = df_country_codes_raw.isnull().sum().sum()

print(f"số mã quốc gia trùng lặp: {dup_codes:,}")
print(f"tổng số giá trị khuyết thiếu: {null_count:,}")

# Kiểm tra độ dài mã chuẩn hóa ISO 3 ký tự
code_lens = df_country_codes_raw["country_code"].astype(str).str.len().unique().tolist()
print(f"độ dài mã quốc gia: {sorted(code_lens)}")
assert sorted(code_lens) == [3], "cảnh báo: có mã quốc gia không đúng 3 ký tự"
assert df_country_codes_raw["country_code"].is_unique, "cảnh báo: mã quốc gia bị trùng lặp"

số mã quốc gia trùng lặp: 0
tổng số giá trị khuyết thiếu: 0
độ dài mã quốc gia: [3]


### Chuẩn hóa tên cột bảng mã quốc gia trong bộ nhớ
Cột region và income_group được gắn tiền tố wb để tránh xung đột với cột chart_region của bảng xếp hạng khi ghép nối. Ô code này đổi tên cột và lưu giữ bảng sạch trong bộ nhớ phục vụ tạo bảng cầu nối.

In [34]:
# Tạo bản sao và đổi tên các trường theo quy ước phân tầng dữ liệu
df_country_codes_clean = df_country_codes_raw.copy()
df_country_codes_clean.rename(
    columns={
        "region": "wb_region",
        "income_group": "wb_income_group"
    },
    inplace=True
)

print(f"đã chuẩn hóa dữ liệu mã quốc gia: {len(df_country_codes_clean):,} dòng trong bộ nhớ")
display(df_country_codes_clean.head(3))

đã chuẩn hóa dữ liệu mã quốc gia: 217 dòng trong bộ nhớ


,country_code,wb_region,wb_income_group
0,ABW,Latin America & Caribbean,High income
1,AFG,South Asia,Low income
2,AGO,Sub-Saharan Africa,Lower middle income


### Nạp dữ liệu thô từ tập tin tổng sản phẩm quốc nội gdp
Tập tin gdp_data.csv ghi nhận giá trị GDP hàng năm của các quốc gia và các nhóm khu vực kinh tế từ năm 1960 đến năm 2022. Ô code này nạp dữ liệu vào biến df_gdp_raw.


In [35]:
from src.data_loader import load_gdp_raw

# Nạp dữ liệu GDP từ nguồn World Bank
raw_gdp_path = "../data/raw/gdp_data.csv"
df_gdp_raw = load_gdp_raw(raw_gdp_path)

print(f"kích thước dữ liệu gdp thô: {df_gdp_raw.shape[0]:,} dòng, {df_gdp_raw.shape[1]} cột")

kích thước dữ liệu gdp thô: 13,365 dòng, 4 cột


### Quan sát các dòng đầu và cuối của bảng dữ liệu gdp thô
Bước quan sát giúp kiểm tra định dạng tên quốc gia, mã ISO, năm và giá trị kinh tế theo đơn vị tiền tệ USD. Ô code này in 3 dòng đầu và 3 dòng cuối của bảng.


In [36]:
print("ba dòng đầu:")
display(df_gdp_raw.head(3))

print("ba dòng cuối:")
display(df_gdp_raw.tail(3))

ba dòng đầu:


,country_name,country_code,year,value
0,Afghanistan,AFG,1960,537777811.1
1,Afghanistan,AFG,1961,548888895.6
2,Afghanistan,AFG,1962,546666677.8


ba dòng cuối:


,country_name,country_code,year,value
13362,Zimbabwe,ZWE,2020,2.150970e+10
13363,Zimbabwe,ZWE,2021,2.837124e+10
13364,Zimbabwe,ZWE,2022,2.067806e+10


### Kiểm tra cấu trúc dữ liệu và thống kê các biến số gdp
Xem xét phân bố của năm ghi nhận và giá trị GDP để nắm bắt khoảng thời gian và quy mô kinh tế. Ô code này in thông tin cấu trúc và bảng thống kê mô tả.


In [37]:
print("thông tin cấu trúc bảng gdp:")
df_gdp_raw.info()

print("thống kê mô tả các biến số:")
display(df_gdp_raw.describe().T)

thông tin cấu trúc bảng gdp:
<class 'pandas.DataFrame'>
RangeIndex: 13365 entries, 0 to 13364
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country_name  13365 non-null  str    
 1   country_code  13365 non-null  str    
 2   year          13365 non-null  int64  
 3   value         13365 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 618.5 KB
thống kê mô tả các biến số:


,count,mean,std,min,25%,50%,75%,max
year,13365.0,1.994479e+03,1.745958e+01,1960.000,1.980000e+03,1.996000e+03,2.009000e+03,2.022000e+03
value,13365.0,1.207051e+12,5.413230e+12,8824746.238,2.523472e+09,1.842778e+10,2.240000e+11,1.010000e+14


### Kiểm tra dòng trùng lặp và tính toàn vẹn cặp quốc gia năm
Mỗi quốc gia trong một năm chỉ được có đúng một giá trị GDP tương ứng để tránh gây nhân bản dòng khi ghép nối. Ô code này kiểm tra dòng trùng lặp hoàn toàn và trùng lặp theo cặp mã quốc gia cùng năm.


In [38]:
# Kiểm tra dòng trùng lặp hoàn toàn và trùng lặp theo cặp khóa
dup_rows = df_gdp_raw.duplicated().sum()
dup_country_year = df_gdp_raw.duplicated(subset=["country_code", "year"]).sum()
null_count = df_gdp_raw.isnull().sum().sum()

print(f"số dòng trùng lặp hoàn toàn: {dup_rows:,}")
print(f"số cặp quốc gia và năm trùng lặp: {dup_country_year:,}")
print(f"tổng số giá trị khuyết thiếu: {null_count:,}")

số dòng trùng lặp hoàn toàn: 0
số cặp quốc gia và năm trùng lặp: 0
tổng số giá trị khuyết thiếu: 0


### Tạo bản sao và chuẩn hóa tên cột giá trị gdp
Cột value được đổi tên thành gdp_usd để phản ánh rõ ý nghĩa đơn vị tiền tệ và mục đích phân tích. Ô code này tạo bản sao và thực hiện đổi tên trường.


In [39]:
# Tạo bản sao dữ liệu sạch và chuẩn hóa tên cột giá trị
df_gdp_clean = df_gdp_raw.copy()
df_gdp_clean.rename(columns={"value": "gdp_usd"}, inplace=True)

print("ba dòng đầu sau khi đổi tên:")
display(df_gdp_clean.head(3))

ba dòng đầu sau khi đổi tên:


,country_name,country_code,year,gdp_usd
0,Afghanistan,AFG,1960,537777811.1
1,Afghanistan,AFG,1961,548888895.6
2,Afghanistan,AFG,1962,546666677.8


### Tách biệt dữ liệu quốc gia độc lập và các nhóm khu vực tổng hợp
Thay vì sử dụng danh sách tên gõ tay, ô code này sử dụng tập 217 mã ISO-3 từ country_codes_clean để phân tách chuẩn xác các quốc gia độc lập và 48 nhóm khu vực tổng hợp vĩ mô.

In [40]:
# Sử dụng tập mã ISO-3 chuẩn từ country_codes_clean để phân tách
valid_country_codes = set(df_country_codes_clean["country_code"].unique())
is_sovereign_country = df_gdp_clean["country_code"].isin(valid_country_codes)

# Tách thành bảng quốc gia độc lập và bảng nhóm tổng hợp vĩ mô
df_gdp_countries = df_gdp_clean[is_sovereign_country].copy()
df_gdp_aggregates = df_gdp_clean[~is_sovereign_country].copy()

# Thống kê các nhóm tổng hợp bị loại bỏ
excluded_codes = df_gdp_aggregates[["country_name", "country_code"]].drop_duplicates()
print(f"số dòng quốc gia độc lập: {len(df_gdp_countries):,} ({df_gdp_countries['country_code'].nunique()} quốc gia)")
print(f"số dòng nhóm tổng hợp vĩ mô: {len(df_gdp_aggregates):,} ({len(excluded_codes)} mã khu vực)")

số dòng quốc gia độc lập: 10,548 (214 quốc gia)
số dòng nhóm tổng hợp vĩ mô: 2,817 (48 mã khu vực)


### Lọc dữ liệu gdp quốc gia theo khung thời gian bảng xếp hạng
Bảng xếp hạng Spotify chỉ quan sát từ cuối năm 2016 đến tháng 7 năm 2022 nên việc lọc khoảng thời gian này giúp tinh gọn bảng dữ liệu cần ghép nối. Ô code này trích xuất các năm từ 2016 đến 2022.


In [41]:
# Lọc dữ liệu gdp các quốc gia trong giai đoạn 2016 đến 2022
n_before = len(df_gdp_countries)
df_gdp_countries_chart_period = df_gdp_countries[
    (df_gdp_countries["year"] >= 2016) & (df_gdp_countries["year"] <= 2022)
].copy()

print(f"số dòng gdp giai đoạn 2016-2022: {len(df_gdp_countries_chart_period):,} (từ {n_before:,})")
print(f"số quốc gia độc nhất trong giai đoạn: {df_gdp_countries_chart_period['country_code'].nunique():,}")

số dòng gdp giai đoạn 2016-2022: 1,436 (từ 10,548)
số quốc gia độc nhất trong giai đoạn: 210


### Tổng kiểm tra các bảng gdp đã làm sạch trong bộ nhớ
Bảng GDP quốc gia hoàn chỉnh và bảng GDP rút gọn theo chu kỳ Spotify được chuẩn hóa và lưu trong bộ nhớ phục vụ kiểm tra chéo độ phủ ở phần sau mà không cần xuất file dư thừa.

In [42]:
# Kiểm tra số lượng quốc gia và dòng dữ liệu gdp trong bộ nhớ
print(f"số dòng gdp quốc gia độc lập: {len(df_gdp_countries):,} dòng ({df_gdp_countries['country_code'].nunique()} quốc gia)")
print(f"số dòng gdp giai đoạn 2016-2022: {len(df_gdp_countries_chart_period):,} dòng ({df_gdp_countries_chart_period['country_code'].nunique()} quốc gia)")
display(df_gdp_countries_chart_period.head(3))

số dòng gdp quốc gia độc lập: 10,548 dòng (214 quốc gia)
số dòng gdp giai đoạn 2016-2022: 1,436 dòng (210 quốc gia)


,country_name,country_code,year,gdp_usd
36,Afghanistan,AFG,2016,1.801955e+10
37,Afghanistan,AFG,2017,1.889635e+10
38,Afghanistan,AFG,2018,1.841886e+10


### Xây dựng và làm giàu bảng cầu nối mã quốc gia country_lookup
Bảng cầu nối country_lookup liên kết 73 quốc gia Spotify với siêu dữ liệu World Bank. Ô code này kết hợp tên chuẩn từ World Bank GDP, bổ sung vùng địa lý, ngôn ngữ và số tuần xếp hạng từ charts_clean, xử lý tường minh Đài Loan và xuất tập tin country_lookup.csv.

In [43]:
# Danh sách 73 quốc gia độc lập từ bảng xếp hạng spotify
spotify_countries = sorted([c for c in df_charts_clean["chart_country"].unique() if c != "Global"])

# Lấy bảng tên quốc gia và mã ISO-3 từ dữ liệu thô World Bank đã nạp
df_wb_names = df_gdp_raw[["country_name", "country_code"]].drop_duplicates()
df_wb_meta = df_wb_names.merge(df_country_codes_clean, on="country_code", how="inner")

# Ánh xạ tên Spotify sang tên World Bank
wb_alias_map = {
    "Czech Republic": "Czechia",
    "Egypt": "Egypt, Arab Rep.",
    "Hong Kong": "Hong Kong SAR, China",
    "Korea": "Korea, Rep.",
    "Slovakia": "Slovak Republic",
    "Turkey": "Turkiye",
    "Venezuela": "Venezuela, RB"
}

lookup_records = []
for sc in spotify_countries:
    target_name = wb_alias_map.get(sc, sc)
    match_row = df_wb_meta[df_wb_meta["country_name"] == target_name]
    if len(match_row) > 0:
        r = match_row.iloc[0]
        lookup_records.append({
            "chart_country": sc,
            "country_code": r["country_code"],
            "wb_country_name": r["country_name"],
            "wb_region": r["wb_region"],
            "wb_income_group": r["wb_income_group"]
        })
    elif sc == "Taiwan":
        # Bổ sung thông tin chuẩn hóa cho Đài Loan do World Bank không báo cáo riêng
        lookup_records.append({
            "chart_country": sc,
            "country_code": "TWN",
            "wb_country_name": "Taiwan, China",
            "wb_region": "East Asia & Pacific",
            "wb_income_group": "High income"
        })
    else:
        raise ValueError(f"phát hiện quốc gia chưa được ánh xạ trong country_lookup: {sc}")

df_country_lookup = pd.DataFrame(lookup_records)

# Làm giàu thêm các trường thông tin khu vực, ngôn ngữ và số tuần có dữ liệu từ bảng xếp hạng
country_meta = df_charts_clean[df_charts_clean["chart_country"] != "Global"].groupby("chart_country", observed=True).agg(
    chart_region=("chart_region", "first"),
    chart_language=("chart_language", "first"),
    n_weeks=("week", "nunique")
).reset_index()

df_country_lookup = df_country_lookup.merge(country_meta, on="chart_country", how="left")

print(f"số quốc gia trong bảng cầu nối: {len(df_country_lookup)}/{len(spotify_countries)}")
assert len(df_country_lookup) == len(spotify_countries), "cảnh báo: thiếu quốc gia trong country_lookup"
assert df_country_lookup["chart_region"].isnull().sum() == 0, "cảnh báo: khuyết thiếu chart_region"

# Xuất bảng cầu nối mã quốc gia hoàn chỉnh
lookup_out_path = "../data/processed/country_lookup.csv"
df_country_lookup.to_csv(lookup_out_path, index=False)
display(df_country_lookup.head(3))

số quốc gia trong bảng cầu nối: 73/73


,chart_country,country_code,wb_country_name,wb_region,wb_income_group,chart_region,chart_language,n_weeks
0,Argentina,ARG,Argentina,Latin America & Caribbean,Upper middle income,South America,Spanish,76
1,Australia,AUS,Australia,East Asia & Pacific,High income,Oceania,English,76
2,Austria,AUT,Austria,Europe & Central Asia,High income,Europe,German,76


### Nạp dữ liệu thô từ tập tin số ca tử vong covid-19
Tập tin global_deaths.csv cung cấp số ca tử vong hàng ngày của các quốc gia trong thời kỳ đại dịch do Đại học Johns Hopkins thu thập. Ô code này nạp dữ liệu thô vào biến df_covid_raw và kiểm tra kích thước bảng ngang ban đầu.


In [44]:
# Nạp dữ liệu tử vong COVID-19 thô từ tập tin csv
raw_covid_path = "../data/raw/global_deaths.csv"
df_covid_raw = pd.read_csv(raw_covid_path)

print(f"kích thước dữ liệu covid thô: {df_covid_raw.shape[0]:,} dòng, {df_covid_raw.shape[1]} cột")
display(df_covid_raw.iloc[:3, :5])

kích thước dữ liệu covid thô: 1,143 dòng, 290 cột


,Country/Region,Afghanistan,Albania,Algeria,Andorra
0,Province/State,NaN,NaN,NaN,NaN
1,1/23/20,0.0,0.0,0.0,0.0
2,1/24/20,0.0,0.0,0.0,0.0


### Chuẩn hóa chuỗi thời gian ngày tháng của dữ liệu covid-19
Dòng đầu tiên chứa nhãn phân cấp tỉnh bang cần được tách riêng trước khi ép kiểu ngày cho toàn bộ các dòng còn lại. Ô code này chuyển đổi chuỗi ngày định dạng tháng ngày năm sang kiểu datetime chuẩn và đặt làm cột đầu tiên.


In [45]:
# Tách dòng phân cấp hành chính và lấy các dòng dữ liệu thời gian
df_covid_dates = df_covid_raw.iloc[1:].copy()

# Ép kiểu datetime cho cột thời gian và sắp xếp lại vị trí cột
df_covid_dates["date"] = pd.to_datetime(df_covid_dates["Country/Region"], format="%m/%d/%y")
df_covid_dates.drop(columns=["Country/Region"], inplace=True)
date_cols = ["date"] + [c for c in df_covid_dates.columns if c != "date"]
df_covid_dates = df_covid_dates[date_cols]

print(f"khoảng thời gian covid: từ {df_covid_dates['date'].min().date()} đến {df_covid_dates['date'].max().date()}")
print(f"tổng số ngày quan sát: {len(df_covid_dates):,}")

khoảng thời gian covid: từ 2020-01-23 đến 2023-03-09
tổng số ngày quan sát: 1,142


### Chuyển đổi bảng dữ liệu covid-19 từ dạng ngang sang dạng dọc
Cấu trúc 290 cột ngang gây khó khăn cho việc ghép nối với dữ liệu bảng xếp hạng theo cặp quốc gia và thời gian. Ô code này chuyển đổi bảng thành 3 cột chuẩn gồm ngày, tên thực thể và số ca tử vong hàng ngày.


In [46]:
# Chuyển đổi bảng ngang thành bảng dọc theo ngày và thực thể địa lý
df_covid_long = df_covid_dates.melt(
    id_vars=["date"],
    var_name="country_raw",
    value_name="daily_deaths"
)

# Ép kiểu số thực an toàn cho số ca tử vong phát sinh hàng ngày
df_covid_long["daily_deaths"] = pd.to_numeric(df_covid_long["daily_deaths"], errors="coerce").fillna(0)

print(f"kích thước bảng dọc sau khi chuyển đổi: {len(df_covid_long):,} dòng, {len(df_covid_long.columns)} cột")
display(df_covid_long.head(3))

kích thước bảng dọc sau khi chuyển đổi: 330,038 dòng, 3 cột


,date,country_raw,daily_deaths
0,2020-01-23,Afghanistan,0.0
1,2020-01-24,Afghanistan,0.0
2,2020-01-25,Afghanistan,0.0


### Hợp nhất dữ liệu cấp tỉnh bang về quốc gia và chuẩn hóa tên gọi
Các quốc gia có phân rã bang như Mỹ, Canada, Úc cần được gộp tổng theo tên quốc gia gốc và chuyển đổi theo bảng tên của Spotify. Ô code này ánh xạ tên quốc gia chuẩn và tính tổng số ca tử vong hàng ngày theo từng nước.


In [47]:
# Hàm chuẩn hóa tên quốc gia và bảo toàn thực thể Hong Kong
def map_country_covid(raw_name):
    if raw_name == "China.12":
        return "Hong Kong"
    base = raw_name.split(".")[0]
    alias = {
        "US": "United States",
        "Czechia": "Czech Republic",
        "Korea, South": "Korea",
        "Taiwan*": "Taiwan"
    }
    return alias.get(base, base)

# Ánh xạ tên chuẩn và tổng hợp số ca tử vong theo quốc gia hàng ngày
df_covid_long["country"] = df_covid_long["country_raw"].apply(map_country_covid)
df_covid_daily = df_covid_long.groupby(["date", "country"], as_index=False)["daily_deaths"].sum()

print(f"số quốc gia độc nhất sau khi gộp bang: {df_covid_daily['country'].nunique():,}")
print(f"kích thước bảng tử vong ngày theo quốc gia: {len(df_covid_daily):,} dòng")

số quốc gia độc nhất sau khi gộp bang: 202
kích thước bảng tử vong ngày theo quốc gia: 230,684 dòng


### Đồng bộ tần suất số ca tử vong theo tuần xếp hạng của spotify
Bảng xếp hạng Spotify cố định ngày chốt tuần vào các ngày thứ Năm nên dữ liệu tử vong hàng ngày cần được gộp thành chu kỳ 7 ngày tương ứng. Ô code này quy đổi mỗi ngày về ngày thứ Năm kết thúc tuần và tính tổng số ca tử vong trong tuần.


In [48]:
# Quy đổi ngày ghi nhận sang ngày thứ Năm kết thúc tuần của Spotify
df_covid_daily["week"] = df_covid_daily["date"].apply(
    lambda d: d + pd.Timedelta(days=(3 - d.weekday()) % 7)
)

# Tính tổng số ca tử vong trong tuần theo từng quốc gia
df_covid_weekly = df_covid_daily.groupby(["country", "week"], as_index=False)["daily_deaths"].sum()
df_covid_weekly.rename(columns={"daily_deaths": "weekly_deaths"}, inplace=True)
df_covid_weekly["weekly_deaths"] = df_covid_weekly["weekly_deaths"].astype(int)

print(f"kích thước bảng tử vong tuần: {len(df_covid_weekly):,} dòng")
display(df_covid_weekly.head(3))

kích thước bảng tử vong tuần: 33,128 dòng


,country,week,weekly_deaths
0,Afghanistan,2020-01-23,0
1,Afghanistan,2020-01-30,0
2,Afghanistan,2020-02-06,0


### Tạo chuỗi tử vong toàn cầu, tính lũy kế và kiểm tra bảo toàn tổng
Ô code này tạo thực thể Global từ tổng số ca tử vong hàng tuần của tất cả các nước, tính chuỗi lũy kế cumulative_deaths và kiểm tra bảo toàn số học tuyệt đối với tổng 6,881,785 ca của dữ liệu gốc.

In [49]:
# Tính tổng số ca tử vong toàn cầu theo từng tuần cho thị trường Global
df_covid_global_weekly = df_covid_weekly.groupby("week", as_index=False)["weekly_deaths"].sum()
df_covid_global_weekly["country"] = "Global"

# Ghép bảng toàn cầu vào bảng quốc gia
df_covid_weekly = pd.concat([df_covid_weekly, df_covid_global_weekly], ignore_index=True)
df_covid_weekly.sort_values(["country", "week"], inplace=True)

# Tính số ca tử vong lũy kế theo thời gian
df_covid_weekly["cumulative_deaths"] = df_covid_weekly.groupby("country")["weekly_deaths"].cumsum()

# Kiểm tra bảo toàn tổng số ca tử vong (phải bằng đúng 6,881,785 ca)
total_country_deaths = df_covid_weekly[df_covid_weekly["country"] != "Global"]["weekly_deaths"].sum()
total_global_deaths = df_covid_weekly[df_covid_weekly["country"] == "Global"]["weekly_deaths"].sum()
print(f"tổng tử vong các quốc gia: {total_country_deaths:,} | tổng tử vong global: {total_global_deaths:,}")
assert total_country_deaths == 6_881_785, "cảnh báo: tổng số ca tử vong không bảo toàn"
assert total_global_deaths == 6_881_785, "cảnh báo: tổng global không khớp tổng quốc gia"


tổng tử vong các quốc gia: 6,881,785 | tổng tử vong global: 6,881,785


### Kiểm tra độ khớp với danh sách quốc gia spotify và xuất tập tin csv
Bước kiểm tra xác nhận tỷ lệ bao phủ của dữ liệu tử vong đối với 74 thực thể trên bảng xếp hạng Spotify. Ô code này kiểm tra độ giao thoa và xuất bảng dữ liệu sạch ra tập tin covid_weekly_clean.csv.


In [50]:
# Lọc bảng tử vong tuần cho đúng 74 thực thể phục vụ phân tích (73 quốc gia và Global)
valid_covid_entities = set(df_country_lookup["chart_country"].unique()) | {"Global"}
df_covid_weekly = df_covid_weekly[df_covid_weekly["country"].isin(valid_covid_entities)].copy()

# Xuất bảng số ca tử vong tuần ra tập tin csv
covid_out_path = "../data/processed/covid_weekly_clean.csv"
df_covid_weekly.to_csv(covid_out_path, index=False)

print(f"đã xuất dữ liệu tử vong tuần: {len(df_covid_weekly):,} dòng ({df_covid_weekly['country'].nunique()} thực thể) ra {covid_out_path}")
display(df_covid_weekly.head(3))

đã xuất dữ liệu tử vong tuần: 12,136 dòng (74 thực thể) ra ../data/processed/covid_weekly_clean.csv


,country,week,weekly_deaths,cumulative_deaths
1148,Argentina,2020-01-23,0,0
1149,Argentina,2020-01-30,0,0
1150,Argentina,2020-02-06,0,0


### Nạp dữ liệu thô từ tập tin báo cáo hạnh phúc thế giới
Tập tin whs_years_updated.csv tổng hợp dữ liệu báo cáo hạnh phúc thế giới từ năm 2015 đến 2024 do Gallup và Liên Hợp Quốc công bố. Ô code này nạp dữ liệu thô vào biến df_whs_raw và kiểm tra số dòng cùng số cột ban đầu.

In [51]:
# Nạp dữ liệu báo cáo hạnh phúc thế giới thô từ tập tin csv
raw_whs_path = "../data/raw/whs_years_updated.csv"
df_whs_raw = pd.read_csv(raw_whs_path)

print(f"kích thước dữ liệu hạnh phúc thô: {df_whs_raw.shape[0]:,} dòng, {df_whs_raw.shape[1]} cột")
display(df_whs_raw.head(3))

# Tạo bản sao làm việc df_whs để bảo toàn dữ liệu thô nguyên vẹn
df_whs = df_whs_raw.copy()


kích thước dữ liệu hạnh phúc thô: 1,510 dòng, 46 cột


,Country,Region,Happiness Rank,Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Year,Lower Confidence Interval,Upper Confidence Interval,Happiness.Rank,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Health..Life.Expectancy.,Trust..Government.Corruption.,Dystopia.Residual,Overall rank,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Perceptions of corruption,Regional indicator,Standard error of ladder score,upperwhisker,lowerwhisker,Logged GDP per capita,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,RANK,Whisker-high,Whisker-low,Dystopia (1.83) + residual,Explained by: GDP per capita
0,Switzerland,Western Europe,1.0,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Iceland,Western Europe,2.0,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Denmark,Western Europe,3.0,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Khảo sát phân bố dữ liệu theo năm và kiểm tra biến mục tiêu
Bộ dữ liệu trải dài qua 10 năm liên tục từ 2015 đến 2024 với biến số cốt lõi là điểm số hạnh phúc. Ô code này kiểm tra số lượng bản ghi theo từng năm và xác nhận tính toàn vẹn không khuyết thiếu của cột điểm số.

In [52]:
# Đếm số lượng quốc gia được khảo sát theo từng năm
yearly_counts = df_whs_raw["Year"].value_counts().sort_index()
print("phân bố số quốc gia theo từng năm:")
display(yearly_counts.to_frame(name="so_quoc_gia"))

# Kiểm tra số lượng giá trị khuyết thiếu của cột điểm hạnh phúc
null_score_count = df_whs_raw["Score"].isnull().sum()
print(f"số lượng giá trị khuyết thiếu của điểm hạnh phúc: {null_score_count}")

phân bố số quốc gia theo từng năm:


,so_quoc_gia
Year,
2015,158
2016,157
2017,155
2018,156
2019,156
2020,153
2021,149
2022,146
2023,137


số lượng giá trị khuyết thiếu của điểm hạnh phúc: 0


### Thống kê mô tả điểm số hạnh phúc và kiểm tra thang đo sáu nhân tố
Ô code này tính toán thống kê mô tả của điểm số hạnh phúc theo từng năm và kiểm tra giá trị trung bình của các nhân tố giải thích để đảm bảo tính đồng nhất thang đo (khoảng 0 đến 1.5).

In [53]:
# Thống kê mô tả điểm số hạnh phúc phân nhóm theo năm
df_score_stats = df_whs_raw.groupby("Year")["Score"].agg(
    so_luong="count",
    nho_nhat="min",
    trung_binh="mean",
    trung_vi="median",
    lon_nhat="max",
    do_lech_chuan="std"
).reset_index()

print("thống kê điểm số hạnh phúc qua các năm:")
display(df_score_stats)

# Kiểm tra giá trị trung bình của 6 nhân tố giải thích theo năm
whs_factor_cols = ["gdp_factor", "social_support", "healthy_life", "freedom", "generosity", "corruption"]
# Chỉ in sau khi các cột đã được hợp nhất ở ô sau


thống kê điểm số hạnh phúc qua các năm:


,Year,so_luong,nho_nhat,trung_binh,trung_vi,lon_nhat,do_lech_chuan
0,2015,158,2.8390,5.375734,5.2325,7.5870,1.145010
1,2016,157,2.9050,5.382185,5.3140,7.5260,1.141674
2,2017,155,2.6930,5.354019,5.2790,7.5370,1.131230
3,2018,156,2.9050,5.375917,5.3780,7.6320,1.119506
4,2019,156,2.8530,5.407096,5.3795,7.7690,1.113120
5,2020,153,2.5669,5.473240,5.5150,7.8087,1.112270
6,2021,149,2.5230,5.532839,5.5340,7.8420,1.073924
7,2022,146,2.4038,5.553575,5.5687,7.8210,1.086814
8,2023,137,1.8590,5.539755,5.6843,7.8042,1.139908
9,2024,143,1.7210,5.527572,5.7853,7.7407,1.170690


### Hợp nhất các cột thứ hạng từ các phiên bản báo cáo
Tên cột thứ hạng hạnh phúc thay đổi qua các năm như Happiness Rank, Happiness.Rank, Overall rank và RANK. Ô code này hợp nhất các cột thứ hạng này thành một cột chung và tính bổ sung cho các năm còn khuyết.

In [54]:
# Hợp nhất các cột thứ hạng sẵn có trong dữ liệu thô
rank_series = (
    df_whs["Happiness Rank"]
    .combine_first(df_whs["Happiness.Rank"])
    .combine_first(df_whs["Overall rank"])
    .combine_first(df_whs["RANK"])
)

# Điền giá trị thứ hạng dựa trên xếp hạng điểm số giảm dần cho các năm còn thiếu
calculated_rank = df_whs.groupby("Year")["Score"].rank(ascending=False, method="min")
df_whs["happiness_rank"] = rank_series.combine_first(calculated_rank).astype(int)

print(f"kiểm tra số giá trị khuyết của thứ hạng: {df_whs['happiness_rank'].isnull().sum()}")
display(df_whs[["Country", "Year", "Score", "happiness_rank"]].head(3))


kiểm tra số giá trị khuyết của thứ hạng: 0


,Country,Year,Score,happiness_rank
0,Switzerland,2015,7.587,1
1,Iceland,2015,7.561,2
2,Denmark,2015,7.527,3


### Hợp nhất sáu nhân tố giải thích chất lượng sống
Báo cáo phân rã điểm số hạnh phúc thành sáu nhân tố giải thích nhưng tên cột có sự khác biệt giữa các giai đoạn báo cáo. Ô code này hợp nhất các cột tương ứng để tạo ra sáu chỉ số thành phần đồng nhất.

In [55]:
# Hợp nhất các cột nhân tố kinh tế gdp bình quân đầu người
gdp_factor = (
    df_whs["Economy (GDP per Capita)"]
    .combine_first(df_whs["Economy..GDP.per.Capita."])
    .combine_first(df_whs["GDP per capita"])
    .combine_first(df_whs["Explained by: Log GDP per capita"])
    .combine_first(df_whs["Explained by: GDP per capita"])
)

# Hợp nhất các cột hỗ trợ xã hội và gia đình
social_factor = (
    df_whs["Explained by: Social support"]
    .combine_first(df_whs["Social support"])
    .combine_first(df_whs["Family"])
)

# Hợp nhất các cột kỳ vọng sống khỏe mạnh
health_factor = (
    df_whs["Explained by: Healthy life expectancy"]
    .combine_first(df_whs["Healthy life expectancy"])
    .combine_first(df_whs["Health (Life Expectancy)"])
    .combine_first(df_whs["Health..Life.Expectancy."])
)

# Gán các cột nhân tố đã hợp nhất vào bảng
df_whs["gdp_factor"] = gdp_factor
df_whs["social_support"] = social_factor
df_whs["healthy_life"] = health_factor
print("đã hợp nhất ba nhân tố: gdp, hỗ trợ xã hội và sức khỏe")


đã hợp nhất ba nhân tố: gdp, hỗ trợ xã hội và sức khỏe


### Hợp nhất các nhân tố tự do, rộng lượng và nhận thức tham nhũng
Ba nhân tố còn lại gồm mức độ tự do lựa chọn cuộc sống, lòng rộng lượng và nhận thức về tham nhũng được ghi nhận dưới các tên cột khác nhau. Ô code này hợp nhất các cột này để hoàn thiện bộ sáu nhân tố chất lượng sống.

In [56]:
# Hợp nhất cột tự do lựa chọn cuộc sống
freedom_factor = (
    df_whs["Explained by: Freedom to make life choices"]
    .combine_first(df_whs["Freedom to make life choices"])
    .combine_first(df_whs["Freedom"])
)

# Hợp nhất cột lòng rộng lượng quyên góp từ thiện
generosity_factor = (
    df_whs["Explained by: Generosity"]
    .combine_first(df_whs["Generosity"])
)

# Hợp nhất cột nhận thức về sự trong sạch và tham nhũng
corruption_factor = (
    df_whs["Explained by: Perceptions of corruption"]
    .combine_first(df_whs["Perceptions of corruption"])
    .combine_first(df_whs["Trust (Government Corruption)"])
    .combine_first(df_whs["Trust..Government.Corruption."])
)

# Gán các cột nhân tố còn lại vào bảng
df_whs["freedom"] = freedom_factor
df_whs["generosity"] = generosity_factor
df_whs["corruption"] = corruption_factor
print("đã hợp nhất đủ sáu nhân tố giải thích hạnh phúc")


đã hợp nhất đủ sáu nhân tố giải thích hạnh phúc


### Trích xuất các trường thông tin chuẩn hóa và tạo bản sao làm sạch
Bảng thô có tới 46 cột dư thừa sau khi hợp nhất. Ô code này trích xuất 10 cột thông tin cốt lõi gồm quốc gia, năm, thứ hạng, điểm hạnh phúc và 6 nhân tố thành phần sang bảng mới df_whs_clean.

In [57]:
# Chọn lọc danh sách cột chuẩn hóa phục vụ phân tích
whs_selected_cols = [
    "Country", "Year", "happiness_rank", "Score",
    "gdp_factor", "social_support", "healthy_life",
    "freedom", "generosity", "corruption"
]

# Tạo bản sao sạch với tên cột dạng chữ thường
df_whs_clean = df_whs[whs_selected_cols].copy()
df_whs_clean.rename(columns={
    "Country": "country",
    "Year": "year",
    "Score": "happiness_score"
}, inplace=True)

print(f"kích thước bảng hạnh phúc sau trích xuất: {len(df_whs_clean):,} dòng, {len(df_whs_clean.columns)} cột")
display(df_whs_clean.head(3))

# Kiểm tra phân bố và thang đo của sáu nhân tố giải thích theo từng năm
factor_means_yearly = df_whs_clean.groupby("year")[["gdp_factor", "social_support", "healthy_life", "freedom", "generosity", "corruption"]].mean()
print("thang đo trung bình của 6 nhân tố giải thích theo từng năm:")
display(factor_means_yearly.round(3))


kích thước bảng hạnh phúc sau trích xuất: 1,510 dòng, 10 cột


,country,year,happiness_rank,happiness_score,gdp_factor,social_support,healthy_life,freedom,generosity,corruption
0,Switzerland,2015,1,7.587,1.39651,1.34951,0.94143,0.66557,0.29678,0.41978
1,Iceland,2015,2,7.561,1.30232,1.40223,0.94784,0.62877,0.43630,0.14145
2,Denmark,2015,3,7.527,1.32548,1.36058,0.87464,0.64938,0.34139,0.48357


thang đo trung bình của 6 nhân tố giải thích theo từng năm:


,gdp_factor,social_support,healthy_life,freedom,generosity,corruption
year,,,,,,
2015,0.846,0.991,0.630,0.429,0.237,0.143
2016,0.954,0.794,0.558,0.371,0.243,0.138
2017,0.985,1.189,0.551,0.409,0.247,0.123
2018,0.891,1.213,0.597,0.455,0.181,0.112
2019,0.905,1.209,0.725,0.393,0.185,0.111
2020,0.869,1.156,0.693,0.464,0.189,0.131
2021,0.977,0.793,0.520,0.499,0.178,0.135
2022,1.410,0.906,0.586,0.517,0.147,0.155
2023,1.407,1.156,0.366,0.540,0.148,0.146


### Chuẩn hóa tên quốc gia và xử lý dòng trùng lặp Cyprus năm 2022
Báo cáo năm 2022 chứa hai dòng mang tên Cyprus do gán nhầm Bắc Síp thành Síp. Ô code này đổi tên dòng có điểm thấp thành North Cyprus, đếm số lượng dòng khớp alias trước khi ánh xạ và chuẩn hóa tên sáu quốc gia theo chuẩn Spotify.

In [58]:
# Đổi tên dòng Síp năm 2022 có điểm số thấp thành North Cyprus
df_whs_clean.loc[(df_whs_clean["country"] == "Cyprus") & (df_whs_clean["year"] == 2022) & (df_whs_clean["happiness_score"] < 6.0), "country"] = "North Cyprus"

# Từ điển chuyển đổi tên quốc gia từ báo cáo hạnh phúc sang chuẩn spotify
whs_country_aliases = {
    "Bolivia, Plurinational State of": "Bolivia",
    "Czechia": "Czech Republic",
    "Korea, Republic of": "Korea",
    "Taiwan, Province of China": "Taiwan",
    "Venezuela, Bolivarian Republic of": "Venezuela",
    "Viet Nam": "Vietnam"
}

# Đếm số lượng dòng khớp với alias trước khi ánh xạ
alias_matched_rows = df_whs_clean["country"].isin(whs_country_aliases.keys()).sum()
print(f"số dòng khớp alias trước khi ánh xạ: {alias_matched_rows} (đúng 6 quốc gia x 10 năm = 60 dòng)")
assert alias_matched_rows == 60, "cảnh báo: số dòng alias không đúng 60"

# Thực hiện ánh xạ tên quốc gia
df_whs_clean["country"] = df_whs_clean["country"].replace(whs_country_aliases)
assert df_whs_clean.duplicated(subset=["country", "year"]).sum() == 0, "cảnh báo: tồn tại trùng lặp cặp quốc gia năm"
print("đã chuẩn hóa tên 6 quốc gia và đảm bảo cặp quốc gia năm duy nhất 100%")

số dòng khớp alias trước khi ánh xạ: 60 (đúng 6 quốc gia x 10 năm = 60 dòng)
đã chuẩn hóa tên 6 quốc gia và đảm bảo cặp quốc gia năm duy nhất 100%


### Khảo sát điểm số hạnh phúc của các quốc gia tiêu biểu
Việc quan sát điểm số hạnh phúc của các quốc gia đặc trưng giúp hiểu rõ phổ phân bố chất lượng sống. Ô code này trích xuất và hiển thị dữ liệu hạnh phúc của nhóm quốc gia Bắc Âu, các cường quốc và Việt Nam.

In [59]:
# Danh sách quốc gia tiêu biểu đại diện cho các nhóm khác nhau
focus_countries = ["Finland", "Denmark", "United States", "United Kingdom", "Japan", "Vietnam"]
df_focus = df_whs_clean[df_whs_clean["country"].isin(focus_countries) & (df_whs_clean["year"] == 2021)]

# Sắp xếp theo thứ hạng hạnh phúc
df_focus = df_focus.sort_values("happiness_rank")
print("điểm số hạnh phúc năm 2021 của các quốc gia tiêu biểu:")
display(df_focus[["country", "happiness_rank", "happiness_score", "gdp_factor", "social_support", "healthy_life"]])

điểm số hạnh phúc năm 2021 của các quốc gia tiêu biểu:


,country,happiness_rank,happiness_score,gdp_factor,social_support,healthy_life
935,Finland,1,7.842,1.446,1.106,0.741
936,Denmark,2,7.620,1.502,1.108,0.763
951,United Kingdom,17,7.064,1.423,1.062,0.757
953,United States,19,6.951,1.533,1.030,0.621
990,Japan,56,5.940,1.389,0.949,0.838
1013,Vietnam,79,5.411,0.817,0.873,0.616


### Lọc dữ liệu theo khung thời gian bảng xếp hạng và kiểm tra toàn vẹn
Bảng xếp hạng Spotify trải dài từ năm 2016 đến 2022 trong khi dữ liệu hạnh phúc có từ 2015 đến 2024. Ô code này phân loại dữ liệu theo khung thời gian Spotify và kiểm tra các giá trị khuyết thiếu còn lại.

In [60]:
# Thống kê số lượng bản ghi thuộc giai đoạn bảng xếp hạng spotify từ 2016 đến 2022
chart_years = list(range(2016, 2023))
df_whs_charts_period = df_whs_clean[df_whs_clean["year"].isin(chart_years)]

print(f"số dòng toàn bộ 10 năm: {len(df_whs_clean):,}")
print(f"số dòng trong giai đoạn spotify (2016-2022): {len(df_whs_charts_period):,}")

# Kiểm tra số lượng giá trị khuyết trên toàn bộ bảng làm sạch
null_summary = df_whs_clean.isnull().sum()
print("số lượng giá trị khuyết thiếu từng cột trên bảng làm sạch:")
display(null_summary.to_frame(name="so_luong_khuyet"))

số dòng toàn bộ 10 năm: 1,510
số dòng trong giai đoạn spotify (2016-2022): 1,072
số lượng giá trị khuyết thiếu từng cột trên bảng làm sạch:


,so_luong_khuyet
country,0
year,0
happiness_rank,0
happiness_score,0
gdp_factor,3
social_support,3
healthy_life,4
freedom,3
generosity,3
corruption,4


### Kiểm tra độ khớp với bảng xếp hạng spotify và xuất tập tin csv
Bước kiểm tra cuối cùng xác nhận 100% các quốc gia thuộc bảng xếp hạng Spotify đều có mặt trong bảng dữ liệu hạnh phúc. Ô code này đối chiếu danh sách thực thể và xuất kết quả ra tập tin whs_clean.csv.

In [61]:
# Đối chiếu danh sách quốc gia giữa dữ liệu hạnh phúc và bảng xếp hạng spotify
spotify_countries = set(df_charts_clean["chart_country"].unique())
whs_countries = set(df_whs_clean["country"].unique())

# Lọc bỏ thực thể Global vì không phải là quốc gia độc lập
spotify_sovereign = spotify_countries - {"Global"}
matched_count = len(spotify_sovereign.intersection(whs_countries))

print(f"độ khớp quốc gia với bảng xếp hạng: {matched_count}/{len(spotify_sovereign)} ({matched_count/len(spotify_sovereign)*100:.1f}%)")
assert matched_count == len(spotify_sovereign), "cảnh báo: có quốc gia trên Spotify chưa khớp dữ liệu hạnh phúc"

# Xuất bảng dữ liệu hạnh phúc sạch ra tập tin csv
whs_out_path = "../data/processed/whs_clean.csv"
df_whs_clean.to_csv(whs_out_path, index=False)
print(f"đã xuất dữ liệu hạnh phúc sạch: {len(df_whs_clean):,} dòng ra {whs_out_path}")

độ khớp quốc gia với bảng xếp hạng: 73/73 (100.0%)
đã xuất dữ liệu hạnh phúc sạch: 1,510 dòng ra ../data/processed/whs_clean.csv


### Kiểm tra ghép nối cặp quốc gia và tuần với dữ liệu tử vong covid-19
Để đảm bảo cấp độ hạt ghép nối chính xác, ô code này thực hiện phép merge có kiểm thử validate="many_to_one" theo cặp (chart_country, week) giữa bảng xếp hạng và covid_weekly_clean, xác nhận tất cả các dòng không khớp đều thuộc giai đoạn trước đại dịch.

In [62]:
# Lấy danh sách các cặp quốc gia và tuần duy nhất từ bảng xếp hạng
df_cw_slots = df_charts_clean[["chart_country", "week"]].drop_duplicates()

# Thực hiện ghép nối với dữ liệu COVID-19 theo cặp khóa (quốc gia, tuần)
m_covid_check = df_cw_slots.merge(
    df_covid_weekly,
    left_on=["chart_country", "week"],
    right_on=["country", "week"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Thống kê kết quả ghép nối
both_count = (m_covid_check["_merge"] == "both").sum()
left_only_count = (m_covid_check["_merge"] == "left_only").sum()
max_left_week = m_covid_check[m_covid_check["_merge"] == "left_only"]["week"].max()

print(f"số cặp quốc gia tuần khớp cả hai bên: {both_count:,}")
print(f"số cặp chỉ có ở chart (trước dịch): {left_only_count:,} (ngày muộn nhất: {max_left_week.date()})")
assert left_only_count == 160, "cảnh báo: số tuần trước dịch khác 160"

# Kiểm tra chặt chẽ toàn bộ 160 tuần trước dịch đều thuộc thị trường Global
left_only = m_covid_check[m_covid_check["_merge"] == "left_only"]
assert (left_only["chart_country"] == "Global").all(), "cảnh báo: có quốc gia khác Global bị thiếu dữ liệu trong left_only"
print("xác nhận 100% các tuần trước dịch đều thuộc Global và các tuần trong đại dịch đều khớp dữ liệu covid-19")

số cặp quốc gia tuần khớp cả hai bên: 5,388
số cặp chỉ có ở chart (trước dịch): 160 (ngày muộn nhất: 2020-01-16)
xác nhận 100% các tuần trước dịch đều thuộc Global và các tuần trong đại dịch đều khớp dữ liệu covid-19


### Kiểm tra ghép nối cặp quốc gia và năm với báo cáo hạnh phúc thế giới
Bảng quốc gia chỉ bao phủ hai năm 2021 và 2022. Ô code này thực hiện kiểm tra ghép nối theo cặp (chart_country, year) với whs_clean bằng validate="many_to_one" để chứng minh độ khớp 100% trong phạm vi phân tích.

In [63]:
# Lọc các cặp quốc gia và năm độc lập (loại bỏ Global)
df_cy_slots = df_charts_country[["chart_country", "year"]].drop_duplicates()

# Ghép nối với dữ liệu hạnh phúc theo cặp khóa (quốc gia, năm)
m_whs_check = df_cy_slots.merge(
    df_whs_clean,
    left_on=["chart_country", "year"],
    right_on=["country", "year"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

whs_both = (m_whs_check["_merge"] == "both").sum()
whs_missing = (m_whs_check["_merge"] == "left_only").sum()
print(f"số cặp quốc gia năm khớp dữ liệu hạnh phúc: {whs_both}/{len(df_cy_slots)} ({whs_both/len(df_cy_slots)*100:.1f}%)")
assert whs_missing == 0, "cảnh báo: có cặp quốc gia năm chưa khớp dữ liệu hạnh phúc"

số cặp quốc gia năm khớp dữ liệu hạnh phúc: 141/141 (100.0%)


### Kiểm tra ghép nối cặp quốc gia và năm với dữ liệu gdp kinh tế vĩ mô
Sử dụng bảng cầu nối country_lookup để chuyển chart_country sang mã ISO-3 và kiểm tra ghép nối với dữ liệu GDP quốc gia theo cặp (country_code, year). Ô code này báo cáo chính xác các trường hợp thiếu dữ liệu do độ trễ báo cáo.

In [64]:
# Ghép mã ISO-3 vào danh sách cặp quốc gia và năm
df_cy_coded = df_cy_slots.merge(df_country_lookup[["chart_country", "country_code"]], on="chart_country", how="left")

# Ghép nối với bảng GDP quốc gia theo cặp khóa (mã quốc gia, năm)
m_gdp_check = df_cy_coded.merge(
    df_gdp_countries,
    on=["country_code", "year"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

gdp_both = (m_gdp_check["_merge"] == "both").sum()
gdp_left_only = m_gdp_check[m_gdp_check["_merge"] == "left_only"]
print(f"số cặp quốc gia năm khớp dữ liệu gdp: {gdp_both}/{len(df_cy_coded)} ({gdp_both/len(df_cy_coded)*100:.1f}%)")
print(f"số cặp thiếu gdp: {len(gdp_left_only)} (gồm Đài Loan không có trong WB và Venezuela năm 2022)")
display(gdp_left_only[["chart_country", "country_code", "year"]])

số cặp quốc gia năm khớp dữ liệu gdp: 138/141 (97.9%)
số cặp thiếu gdp: 3 (gồm Đài Loan không có trong WB và Venezuela năm 2022)


,chart_country,country_code,year
122,Taiwan,TWN,2021
123,Taiwan,TWN,2022
138,Venezuela,VEN,2022


## PHẦN 9: TỔNG HỢP VÀ XUẤT CÁC BẢNG DATA MARTS PHỤC VỤ TRỰC QUAN HÓA VÀ MÔ HÌNH HÓA
Để tối ưu hóa hiệu năng cho các bảng điều khiển trực quan (Dashboard 1, 2, 3) và bài toán Machine Learning (Regression, Classification), phần này xây dựng 4 bảng dữ liệu tổng hợp tinh gọn từ các bảng sạch đã liên kết chuẩn hóa.

### Xây dựng Data Mart xu hướng âm thanh theo tuần mart_audio_trend_weekly
Bảng mart_audio_trend_weekly phục vụ Dashboard 1 và Dashboard 3 với cấp hạt (chart_country, week). Bảng tính toán cả chỉ số trung bình thường và trung bình có trọng số lượt nghe (streams-weighted) cho 12 đặc trưng âm học sau khi đã loại bỏ nhân bản do nghệ sĩ hợp tác.

In [65]:
# Deduplicate slot vị trí xếp hạng để tránh nhân bản trọng số nghệ sĩ hợp tác
df_slots = df_charts_clean.drop_duplicates(subset=["chart_country", "week", "rank"])

# Danh sách 12 đặc trưng âm học
audio_feature_cols = [
    "danceability", "energy", "key", "mode", "loudness", 
    "speechiness", "acousticness", "instrumentalness", 
    "liveness", "valence", "tempo", "duration"
]

# Hàm tổng hợp đặc trưng âm học theo tuần
def agg_audio_weekly(g):
    tot_streams = g["streams"].sum()
    res = {
        "total_streams": tot_streams,
        "track_count": len(g),
        "new_entries": int(g["is_new_entry"].sum())
    }
    for col in audio_feature_cols:
        res[f"{col}_mean"] = g[col].mean()
        res[f"{col}_weighted"] = (g[col] * g["streams"]).sum() / tot_streams if tot_streams > 0 else g[col].mean()
    return pd.Series(res)

# Tổng hợp theo quốc gia và tuần
df_mart_audio_weekly = df_slots.groupby(["chart_country", "week"], observed=True).apply(agg_audio_weekly).reset_index()

# Xuất ra định dạng parquet tối ưu hiệu năng đọc
mart_audio_out_path = "../data/processed/mart_audio_trend_weekly.parquet"
df_mart_audio_weekly.to_parquet(mart_audio_out_path, index=False)

print(f"đã xuất mart_audio_trend_weekly.parquet: {len(df_mart_audio_weekly):,} dòng, {len(df_mart_audio_weekly.columns)} cột")
display(df_mart_audio_weekly.head(3))

đã xuất mart_audio_trend_weekly.parquet: 5,548 dòng, 29 cột


,chart_country,week,total_streams,track_count,new_entries,danceability_mean,danceability_weighted,energy_mean,energy_weighted,key_mean,key_weighted,mode_mean,mode_weighted,loudness_mean,loudness_weighted,speechiness_mean,speechiness_weighted,acousticness_mean,acousticness_weighted,instrumentalness_mean,instrumentalness_weighted,liveness_mean,liveness_weighted,valence_mean,valence_weighted,tempo_mean,tempo_weighted,duration_mean,duration_weighted
0,Argentina,2021-02-04,96192710.0,200.0,16.0,0.71143,0.735600,0.657585,0.659894,5.23,5.215540,0.525,0.496514,-5.661310,-5.377115,0.114690,0.134070,0.232841,0.232843,0.003564,0.001755,0.167194,0.151609,0.632594,0.655807,126.869453,129.344360,214014.765625,206121.042976
1,Argentina,2021-02-11,98730688.0,200.0,13.0,0.71567,0.736972,0.663280,0.661955,5.23,5.219331,0.550,0.508641,-5.597865,-5.359143,0.115112,0.132871,0.231033,0.229692,0.003656,0.001859,0.170701,0.155172,0.642103,0.660159,125.213287,128.379879,216345.093750,207608.310346
2,Argentina,2021-02-18,96438610.0,200.0,10.0,0.71548,0.735207,0.668385,0.663016,5.28,5.285699,0.550,0.515347,-5.543840,-5.340677,0.116869,0.134443,0.225014,0.228221,0.003658,0.001860,0.172039,0.161561,0.642410,0.660596,125.374275,127.782180,214453.140625,207663.596013


### Xây dựng Data Mart phân tích nghệ sĩ mart_artist_chart
Bảng mart_artist_chart phục vụ Dashboard 1 với cấp hạt artist_id. Bảng tổng hợp tổng lượt stream trên bảng xếp hạng, số ca khúc lọt bảng, thứ hạng cao nhất, số tuần xuất hiện, cùng tên nghệ sĩ và thể loại âm nhạc được lấy trực tiếp từ bảng xếp hạng sạch, bảo đảm tỷ lệ đầy đủ 100% không khuyết thiếu.

In [66]:
# Tổng hợp chỉ số thành tích trên bảng xếp hạng của từng nghệ sĩ từ charts_clean
# Phân tách Global và Country để tránh đếm trùng lượt streams và ghi nhận đúng tên nghệ sĩ độc lập

df_global = df_charts_clean[df_charts_clean['chart_country'] == 'Global']
df_country = df_charts_clean[df_charts_clean['chart_country'] != 'Global']

artist_base = df_charts_clean.groupby('artist_id', observed=True).agg(
    artist_name=('artist_individual', 'first'),
    artist_genre=('artist_genre', 'first'),
    n_chart_tracks=('track_id', 'nunique')
).reset_index()

global_agg = df_global.groupby('artist_id', observed=True).agg(
    global_streams=('streams', 'sum'),
    global_peak_rank=('rank', 'min'),
    global_weeks_on_chart=('week', 'nunique')
).reset_index()

country_agg = df_country.groupby('artist_id', observed=True).agg(
    country_streams=('streams', 'sum'),
    country_peak_rank=('rank', 'min'),
    country_weeks_on_chart=('week', 'nunique')
).reset_index()

df_mart_artist = artist_base.merge(global_agg, on='artist_id', how='left').merge(country_agg, on='artist_id', how='left')
df_mart_artist['global_streams'] = df_mart_artist['global_streams'].fillna(0).astype('int64')
df_mart_artist['country_streams'] = df_mart_artist['country_streams'].fillna(0).astype('int64')
df_mart_artist['global_weeks_on_chart'] = df_mart_artist['global_weeks_on_chart'].fillna(0).astype('int64')
df_mart_artist['country_weeks_on_chart'] = df_mart_artist['country_weeks_on_chart'].fillna(0).astype('int64')
df_mart_artist['global_peak_rank'] = df_mart_artist['global_peak_rank'].fillna(-1).astype('int64')
df_mart_artist['country_peak_rank'] = df_mart_artist['country_peak_rank'].fillna(-1).astype('int64')

# Xuất dữ liệu nghệ sĩ xếp hạng ra định dạng parquet
mart_artist_out_path = "../data/processed/mart_artist_chart.parquet"
df_mart_artist.to_parquet(mart_artist_out_path, index=False)

print(f"đã xuất mart_artist_chart.parquet: {len(df_mart_artist):,} nghệ sĩ, {len(df_mart_artist.columns)} cột (đầy đủ 100% tên nghệ sĩ)")
display(df_mart_artist.head(3))


đã xuất mart_artist_chart.parquet: 15,254 nghệ sĩ, 10 cột (đầy đủ 100% tên nghệ sĩ)


,artist_id,artist_name,artist_genre,n_chart_tracks,global_streams,global_peak_rank,global_weeks_on_chart,country_streams,country_peak_rank,country_weeks_on_chart
0,000TJhsWOqbyNbTqCU26cW,KlikKlak,unknown,1,0,-1,0,297007,69,1
1,001oyhl94fb7YKL6fVqHOo,Opiat,unknown,10,0,-1,0,28907060,9,76
2,004cRPgBOiWUUsu3mXXd1W,RAAMIS,unknown,1,0,-1,0,9745,200,1


### Xây dựng Data Mart tương quan âm thanh và hạnh phúc mart_country_year
Bảng mart_country_year phục vụ Dashboard 2 với cấp hạt (chart_country, year). Bảng kết hợp các đặc trưng âm học năm của 73 quốc gia với điểm số hạnh phúc thế giới whs_clean và siêu dữ liệu địa lý kinh tế country_lookup.

In [67]:
# Lọc các vị trí xếp hạng của 73 quốc gia độc lập (loại trừ Global)
df_slots_country = df_slots[df_slots["chart_country"] != "Global"].copy()

# Hàm tổng hợp đặc trưng âm học hàng năm theo quốc gia
def agg_audio_yearly(g):
    tot_streams = g["streams"].sum()
    res = {
        "yearly_streams": tot_streams,
        "slot_count": len(g),
        "n_weeks": g["week"].nunique()
    }
    for col in audio_feature_cols:
        res[f"{col}_mean"] = g[col].mean()
        res[f"{col}_weighted"] = (g[col] * g["streams"]).sum() / tot_streams if tot_streams > 0 else g[col].mean()
    return pd.Series(res)

df_country_year_audio = df_slots_country.groupby(["chart_country", "year"], observed=True).apply(agg_audio_yearly).reset_index()

# Ghép siêu dữ liệu phân vùng và nhóm thu nhập từ country_lookup
df_mart_country_year = df_country_year_audio.merge(
    df_country_lookup[["chart_country", "country_code", "wb_region", "wb_income_group", "chart_region", "chart_language"]],
    on="chart_country",
    how="inner"
)

# Ghép chỉ số hạnh phúc thế giới từ whs_clean
df_mart_country_year = df_mart_country_year.merge(
    df_whs_clean,
    left_on=["chart_country", "year"],
    right_on=["country", "year"],
    how="inner"
)
df_mart_country_year.drop(columns=["country"], inplace=True)

# Xuất bảng tương quan quốc gia năm
mart_cy_out_path = "../data/processed/mart_country_year.parquet"
df_mart_country_year.to_parquet(mart_cy_out_path, index=False)

print(f"đã xuất mart_country_year.parquet: {len(df_mart_country_year)} dòng, {len(df_mart_country_year.columns)} cột (khớp đúng 141 cặp)")
display(df_mart_country_year.head(3))

đã xuất mart_country_year.parquet: 141 dòng, 42 cột (khớp đúng 141 cặp)


,chart_country,year,yearly_streams,slot_count,n_weeks,danceability_mean,danceability_weighted,energy_mean,energy_weighted,key_mean,key_weighted,mode_mean,mode_weighted,loudness_mean,loudness_weighted,speechiness_mean,speechiness_weighted,acousticness_mean,acousticness_weighted,instrumentalness_mean,instrumentalness_weighted,liveness_mean,liveness_weighted,valence_mean,valence_weighted,tempo_mean,tempo_weighted,duration_mean,duration_weighted,country_code,wb_region,wb_income_group,chart_region,chart_language,happiness_rank,happiness_score,gdp_factor,social_support,healthy_life,freedom,generosity,corruption
0,Argentina,2021,4.899018e+09,9598.0,48.0,0.714251,0.730558,0.670573,0.666169,5.207752,5.084101,0.545426,0.540644,-5.454952,-5.227093,0.111467,0.116309,0.238378,0.240095,0.003588,0.001887,0.168837,0.161242,0.634624,0.640379,123.141525,122.953834,211607.390625,206567.139543,ARG,Latin America & Caribbean,Upper middle income,South America,Spanish,57,5.929,1.162000,0.980000,0.646000,0.544000,0.069000,0.067000
1,Argentina,2022,3.084508e+09,5582.0,28.0,0.723632,0.731713,0.679952,0.673263,5.040129,5.064631,0.530634,0.506177,-5.372862,-5.224030,0.109831,0.113715,0.236349,0.236340,0.003150,0.002642,0.165618,0.162965,0.630108,0.621297,120.192291,117.627724,203098.125000,200553.237097,ARG,Latin America & Caribbean,Upper middle income,South America,Spanish,57,5.967,1.591656,1.101976,0.661976,0.555376,0.080515,0.084926
2,Australia,2021,3.853955e+09,9600.0,48.0,0.654205,0.659392,0.628322,0.627767,5.122187,5.165501,0.660417,0.667641,-6.508526,-6.465723,0.096267,0.094968,0.262439,0.262947,0.006933,0.005297,0.165630,0.165859,0.513179,0.517641,119.311943,120.753474,205640.437500,201932.248839,AUS,East Asia & Pacific,High income,Oceania,English,11,7.183,1.453000,1.076000,0.801000,0.647000,0.291000,0.317000


### Xây dựng tập dữ liệu mô hình hóa ml_tracks và biến mục tiêu pop_score
Bảng ml_tracks phục vụ huấn luyện mô hình Machine Learning với cấp hạt track_id duy nhất (39,190 bài hát). Biến mục tiêu pop_score (0–100) được tính toán đa chiều từ thứ hạng đỉnh cao, sức bền, độ phủ quốc gia và lượt nghe log-scale. Toàn bộ đặc trưng đầu vào X được trích xuất sạch sẽ từ tracks_dimension (12 chỉ số âm học nội tại, artists_num, artist_genre, release_year, release_month), bảo đảm 0% giá trị khuyết thiếu và không bị rò rỉ dữ liệu.

In [68]:
import numpy as np

# Đếm số quốc gia độc lập xuất hiện (loại trừ Global, mẫu số chuẩn là 73)
country_slots = df_slots[df_slots["chart_country"] != "Global"]
country_counts = country_slots.groupby("track_id")["chart_country"].nunique().rename("n_countries")

# Tổng hợp chỉ số bài hát từ các slot không trùng lặp
tracks_agg = df_slots.groupby("track_id").agg(
    peak_rank=("rank", "min"),
    nweek=("week", "nunique"),
    total_streams=("streams", "sum")
).join(country_counts).fillna({"n_countries": 0}).reset_index()

# Chuẩn hóa từng thành phần về thang đo [0, 1]
tracks_agg["s_rank"] = 1.0 - (tracks_agg["peak_rank"] - 1.0) / 199.0
cap_weeks = tracks_agg["nweek"].quantile(0.99)
tracks_agg["s_weeks"] = tracks_agg["nweek"].clip(upper=cap_weeks) / cap_weeks
tracks_agg["s_countries"] = tracks_agg["n_countries"] / 73.0
log_str = np.log1p(tracks_agg["total_streams"])
tracks_agg["s_streams"] = (log_str - log_str.min()) / (log_str.max() - log_str.min())

# Tính pop_score theo 4 trọng số cân bằng (thang điểm 0 đến 100)
W_RANK, W_WEEKS, W_COUNTRIES, W_STREAMS = 0.30, 0.25, 0.20, 0.25
tracks_agg["pop_score"] = (
    W_RANK      * tracks_agg["s_rank"] +
    W_WEEKS     * tracks_agg["s_weeks"] +
    W_COUNTRIES * tracks_agg["s_countries"] +
    W_STREAMS   * tracks_agg["s_streams"]
) * 100.0

# Phái sinh biến mục tiêu phân loại nhị phân và phân nhóm
# is_hit: ngưỡng 50 điểm (đại diện Top 13.9% bài hát thành công vượt trội, cân bằng tốt cho Logistic Regression)
# is_mega_hit: ngưỡng 70 điểm (đại diện Top 1.6% hiện tượng toàn cầu)
tracks_agg["is_hit"] = (tracks_agg["pop_score"] >= 50.0).astype(int)
tracks_agg["is_mega_hit"] = (tracks_agg["pop_score"] >= 70.0).astype(int)
tracks_agg["hit_tier"] = pd.cut(
    tracks_agg["pop_score"],
    bins=[0, 25, 50, 75, 100],
    labels=["Niche", "Moderate", "Popular", "Mega Hit"],
    include_lowest=True
)

# Ghép với 12 đặc trưng âm học nội tại và thông tin phát hành từ tracks_dimension
df_ml = df_tracks.merge(
    tracks_agg[["track_id", "pop_score", "is_hit", "is_mega_hit", "hit_tier", "peak_rank", "nweek", "total_streams", "n_countries"]],
    on="track_id",
    how="inner"
)

# Trích xuất thêm đặc trưng tháng phát hành
df_ml["release_month"] = pd.to_datetime(df_ml["release_date_clean"]).dt.month.astype("Int8")

# Xuất ra cả định dạng parquet và csv
df_ml.to_parquet("../data/processed/ml_tracks.parquet", index=False)
df_ml.to_csv("../data/processed/ml_tracks.csv", index=False)

print(f"đã xuất ml_tracks.parquet và csv: {len(df_ml):,} bài hát, {len(df_ml.columns)} cột (0% khuyết thiếu)")
display(df_ml[["track_name", "pop_score", "is_hit", "is_mega_hit", "hit_tier", "n_countries", "total_streams"]].head(3))

đã xuất ml_tracks.parquet và csv: 39,190 bài hát, 30 cột (0% khuyết thiếu)


,track_name,pop_score,is_hit,is_mega_hit,hit_tier,n_countries,total_streams
0,Plan A,61.496542,1,0,Popular,20.0,255207049
1,ARRANCARMELO,53.982470,1,0,Popular,10.0,26337546
2,Chance,57.516414,1,0,Popular,18.0,61927262
